# Controlled experiments: NLI / MCS targets → GCN / R-GCN / GAT
Run shared setup once, then each experiment cell independently, in any order.
Each experiment cell builds its targets if needed, trains five seeds, evaluates, and saves its results.

Inputs match the other notebooks: `/home/svelag11/agentic/results.json`,
`steps_to_annotate_all.csv`, and shared `nli_cache/`. All new outputs go to
`/home/svelag11/agentic/gibs_controlled_experiments/`; existing checkpoints are untouched.

**Controls:** same 15 training tasks, 9 evaluation tasks, operation-text MiniLM features,
2 NLI features, hidden size 64, two layers, weighted entropy+BCE loss (λ=2),
AdamW, 50 epochs, cosine schedule, gradient clipping, and seeds 0–4.
NLI targets are the original recipe's **global maximum**, not Eq. 2's per-run average.
LOO excludes the query run. MCS targets are the mean node-inclusion frequency over
other passing runs, using the existing **heuristic top-k greedy MCS matcher**, not an exact solver.
MCS uses operation, result, and source context; changing targets therefore changes the supervision signal.

GCN merges undirected temporal and dependency edges. R-GCN keeps those same two types
separate. Masked GAT attends over their union. Complete GAT attends over all step pairs,
including future steps: this is offline diagnosis. Only the complete-GAT **forward pass**
avoids heuristic connectivity; MCS targets still use heuristic graphs.
Attention heads are averaged; all architectures use a simple linear prediction head.
Parameter counts are reported because architectures are not parameter-matched.

The settings are fixed for comparison with the reproduction. Evaluation labels are used
only for reporting; do not choose hyperparameters from these test results. If tuning next,
reserve validation tasks and retrain every comparator on the same reduced training split.
No training results are prefilled.

**Additional experiments 09–13:** directional NLI edges replace value-matching
agent-to-agent dependency links, with temporal links and NLI training targets fixed.
Hard NLI keeps the highest-scoring earlier agent step if entailment ≥0.3;
weighted NLI keeps every earlier agent step with its continuous entailment probability.
Both are symmetrized for message passing to match the original offline GCN.
NLI measures support, not verified causal dependency; the hard cutoff remains a heuristic.
User turns remain represented by the same user-alignment input feature.

Experiment 13 uses **approximate, time-limited GED alignment** to derive LOO step targets.
GED selects a node mapping using node and edge edit costs. A matched query step receives
its semantic similarity to the mapped reference step; a deleted query step receives zero.
Scores are averaged across other passing runs. This is an experimental target, not a
paper method or a calibrated error probability. It uses the same operation/source/result
projection as the MCS matcher, with typed dependency/temporal edges.
Run 13 separately: GED is CPU-intensive. Each pair is cached; solver time is capped at
2 seconds per pair (excluding preparation). A valid all-delete/all-insert path is the
fallback if no candidate is yielded; fallback counts and pair timings are saved.
Neither a timeout nor a returned mapping guarantees an optimal GED solution.


**Legacy caches:** reuses the shared NLI cache automatically. For MCS-LOO and NLI-LOO,
looks for `gibs_nli/mcs_targets_task*.json` and `gibs_nli/soft_targets_loo_task*.json`.
Every task must have exact run/step coverage and finite scores in [0,1]. It then
recomputes up to two query runs per task (passing/failing where available) and rejects
that task's cache if targets disagree. This is a **sample audit, not full verification**;
old files do not contain enough metadata to establish provenance. Decisions are saved
to `legacy_cache_audit.json`. Disable `REUSE_LEGACY_TARGETS` to rebuild instead.
Original JSON files are read only. Anonymous pickle training data and old model weights
are not imported. Accepted targets are saved under the new notebook's cache signature.


In [1]:
# Shared setup 1: packages, paths, reproducibility
import importlib.util, subprocess, sys
packages = {'torch':'torch', 'transformers':'transformers', 'numpy':'numpy',
            'pandas':'pandas', 'sklearn':'scikit-learn', 'tqdm':'tqdm',
            'huggingface_hub':'huggingface-hub', 'accelerate':'accelerate', 'networkx':'networkx', 'scipy':'scipy'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import os, json, csv, random, math, hashlib, inspect, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DRIVE_BASE = '/home/svelag11/agentic'
RESULTS = Path(DRIVE_BASE) / 'results.json'
ANNOT_CSV = Path(DRIVE_BASE) / 'steps_to_annotate_all.csv'
CACHE_DIR = Path(DRIVE_BASE) / 'nli_cache'
OUT_DIR = Path(DRIVE_BASE) / 'gibs_controlled_experiments'
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['STEPWISE_NLI_CACHE'] = str(CACHE_DIR)
TRAIN_TASKS = [16,17,20,21,22,23,24,32,33,35,37,39,40,41,42]
EVAL_TASKS = [1,2,9,11,12,15,18,43,47]
SEEDS = [0,1,2,3,4]
EPOCHS, LR, HIDDEN, HEADS, LAM = 50, 3e-4, 64, 4, 2.0
FORCE_RETRAIN = False
assert not set(TRAIN_TASKS) & set(EVAL_TASKS)
for path in [RESULTS, ANNOT_CSV]:
    assert path.is_file(), f'Missing input: {path}'
def file_digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
DATA_HASH = file_digest(RESULTS)
print(f'Device: {DEVICE} | Outputs: {OUT_DIR}')

Device: cuda | Outputs: /home/svelag11/agentic/gibs_controlled_experiments


## Shared definitions and models

In [2]:
# ── gibs/graph/step_graph.py (inlined) ───────────────────────────────────────
import json, re
from typing import Any

ENUM_VALUES = {"yes", "no", "true", "false", "economy", "business", "basic_economy", "one_way", "round_trip"}
VALUE_TOKEN = re.compile(
    r"\d{4}-\d{2}-\d{2}"
    r"|\b[A-Z0-9]{6}\b"
    r"|\b[a-z]+_[a-z]+_\d+\b"
    r"|\b(?:gift_card|credit_card|certificate)_\d+\b"
    r"|\bHAT\d{3}\b"
)
FREE_TEXT_MIN_LEN = 40
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
CONTROL_TOKEN = re.compile(r"^###([A-Z_\-]+)###$")
CONTROL_TEXT = {
    "STOP": "The user ended the conversation.",
    "TRANSFER": "The user was transferred to a human agent.",
    "OUT-OF-SCOPE": "The user said the request is out of scope.",
}
UNICODE_FIXES = {"\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2013": "-", "\u2014": "-",
                 "\u00a0": " ", "\u202f": " ", "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"'}

def normalize_text(text):
    for k, v in UNICODE_FIXES.items():
        text = text.replace(k, v)
    text = text.replace("**", "").replace("__", "")
    return re.sub(r"\s+", " ", text).strip()

def _flatten(value, prefix=""):
    if isinstance(value, dict):
        out = []
        for k, v in value.items():
            out += _flatten(v, f"{prefix}.{k}" if prefix else str(k))
        return out
    if isinstance(value, list):
        if not value:
            return [(prefix, "[]")]
        out = []
        for i, v in enumerate(value):
            out += _flatten(v, f"{prefix}[{i}]")
        return out
    return [(prefix, value)]

def _scalar(v):
    return "null" if v is None else (str(v).lower() if isinstance(v, bool) else str(v))

def verbalize_tool_call(name, arguments):
    pairs = _flatten(arguments or {})
    if not pairs:
        return f"Call {name} with no arguments."
    return f"Call {name} with " + "; ".join(f"{k} = {normalize_text(_scalar(v))}" for k, v in pairs) + "."

def verbalize_observation(content, error):
    if error:
        return "Tool error: " + normalize_text(content)
    try:
        parsed = json.loads(content)
    except (json.JSONDecodeError, TypeError):
        return "Tool result: " + normalize_text(str(content))
    if isinstance(parsed, (dict, list)):
        return "Tool result: " + "; ".join(f"{k}: {normalize_text(_scalar(v))}" for k, v in _flatten(parsed)) + "."
    return "Tool result: " + normalize_text(_scalar(parsed))

def verbalize_utterance(content):
    text = (content or "").strip()
    m = CONTROL_TOKEN.match(text)
    if m:
        return CONTROL_TEXT.get(m.group(1), f"The user sent the control signal {m.group(1)}.")
    return normalize_text(text)

def _reasoning(msg):
    try:
        m = msg["raw_data"]["choices"][0]["message"]
    except (KeyError, IndexError, TypeError):
        return None
    return m.get("reasoning") or m.get("reasoning_content") or None

def _leaves(v):
    if isinstance(v, dict):
        for x in v.values(): yield from _leaves(x)
    elif isinstance(v, list):
        for x in v: yield from _leaves(x)
    else:
        yield v

def dependency_values(arguments):
    out = []
    for v in _leaves(arguments or {}):
        if not isinstance(v, str):
            continue
        if len(v) >= FREE_TEXT_MIN_LEN:
            cands = VALUE_TOKEN.findall(v)
        elif len(v) < 3 or v.lower() in ENUM_VALUES:
            cands = []
        else:
            cands = [v]
        for c in cands:
            if c not in out:
                out.append(c)
    return out

def value_occurs(value, text):
    left = r"(?<![A-Za-z0-9])"
    right = r"(?![A-Za-z0-9])" if not ISO_DATE.match(value) else r"(?![A-SU-Za-z0-9])"
    return re.search(left + re.escape(value) + right, text) is not None

def _visible_text(node):
    if node["type"] == "UserTurn":
        return node.get("content") or ""
    if node.get("kind") == "tool_call":
        return node["observation"]["content_raw"] or ""
    return node.get("content") or ""

def build_step_graph(sim, info=None, task=None, source_file=None):
    msgs = sim.get("messages") or []
    nodes = []
    step_index = 0
    i = 0
    while i < len(msgs):
        m = msgs[i]
        role = m["role"]
        if role == "user":
            nodes.append({
                "id": f"u{i}", "type": "UserTurn", "scorable": False,
                "raw_message_indices": [i], "turn_idx": m.get("turn_idx"), "timestamp": m.get("timestamp"),
                "content": m.get("content"),
                "op_text": verbalize_utterance(m.get("content")),
                "result_text": verbalize_utterance(m.get("content")),
            })
        elif role == "assistant":
            node = {
                "id": f"s{i}", "type": "AgentStep", "scorable": True, "step_index": step_index,
                "raw_message_indices": [i], "turn_idx": m.get("turn_idx"), "timestamp": m.get("timestamp"),
                "reasoning": _reasoning(m), "content": m.get("content"),
            }
            calls = m.get("tool_calls") or []
            if len(calls) > 1:
                raise ValueError(f"message {i}: parallel tool calls not supported")
            if calls:
                tc = calls[0]
                if i + 1 >= len(msgs) or msgs[i+1]["role"] != "tool" or msgs[i+1].get("id") != tc["id"]:
                    raise ValueError(f"message {i}: tool call {tc['id']} has no immediately following observation")
                obs = msgs[i+1]
                try:
                    parsed = json.loads(obs["content"])
                except (json.JSONDecodeError, TypeError):
                    parsed = None
                node.update({
                    "kind": "tool_call",
                    "tool_call": {"id": tc["id"], "name": tc["name"], "arguments": tc.get("arguments") or {}},
                    "observation": {"tool_call_id": obs["id"], "error": bool(obs.get("error")),
                                    "raw_message_index": i+1, "timestamp": obs.get("timestamp"),
                                    "content_raw": obs["content"], "content_parsed": parsed},
                    "op_text": verbalize_tool_call(tc["name"], tc.get("arguments") or {}),
                    "result_text": verbalize_observation(obs["content"], bool(obs.get("error"))),
                })
                node["raw_message_indices"].append(i+1)
                i += 1
            else:
                node.update({"kind": "message",
                             "op_text": verbalize_utterance(m.get("content")),
                             "result_text": verbalize_utterance(m.get("content"))})
            nodes.append(node)
            step_index += 1
        else:
            raise ValueError(f"message {i}: unpaired role {role!r}")
        i += 1

    for k, n in enumerate(nodes):
        if n["type"] != "AgentStep": continue
        if n["kind"] == "tool_call":
            n["response_node"] = n["id"]
        else:
            nxt = nodes[k+1] if k+1 < len(nodes) else None
            n["response_node"] = nxt["id"] if nxt and nxt["type"] == "UserTurn" else None

    edges = [{"id": f"t{k}", "type": "temporal", "source": a["id"], "target": b["id"]}
             for k, (a, b) in enumerate(zip(nodes, nodes[1:]))]
    dep_count = 0
    for k, n in enumerate(nodes):
        if n.get("kind") != "tool_call": continue
        grouped = {}
        for value in dependency_values(n["tool_call"]["arguments"]):
            hits = [p["id"] for p in nodes[:k] if value_occurs(value, _visible_text(p))]
            if not hits: continue
            src = hits[0]
            g = grouped.setdefault(src, {"values": [], "alternative_sources": {}})
            g["values"].append(value)
            if hits[1:]: g["alternative_sources"][value] = hits[1:]
        for src, g in grouped.items():
            edges.append({"id": f"d{dep_count}", "type": "data_dependency", "source": src, "target": n["id"],
                          "values": g["values"], "alternative_sources": g["alternative_sources"],
                          "rule": "earliest prior node whose raw visible text contains the value"})
            dep_count += 1

    info = info or {}
    ri = sim.get("reward_info") or {}
    reward = ri.get("reward")
    metadata = {
        "source_file": source_file, "simulation_id": sim.get("id"),
        "task_id": str(sim.get("task_id")), "trial": sim.get("trial"),
        "reward": reward, "success": None if reward is None else reward >= 0.5,
    }
    return {"schema_version": "0.2",
            "graph_id": f"task{metadata['task_id']}_{str(sim.get('id'))[:8]}",
            "metadata": metadata, "nodes": nodes, "edges": edges}

print('step_graph ready')

step_graph ready


In [3]:
# ── gibs/mcs/nli.py (inlined, essential parts only) ──────────────────────────
import hashlib, os
from pathlib import Path

DEFAULT_CACHE_DIR = Path(os.environ.get("STEPWISE_NLI_CACHE", Path.home() / ".cache" / "stepwise_nli"))

def symmetric_input_ids(tokenizer, a, b, max_length):
    full = tokenizer(f"{a} [SEP] {b}", truncation=False)["input_ids"]
    cls, sep = tokenizer.cls_token_id, tokenizer.sep_token_id
    mid = full.index(sep, 1)
    end = mid
    while full[end+1] == sep and end+1 < len(full)-1:
        end += 1
    sep_block = full[mid:end+1]
    ids_a, ids_b = full[1:mid], full[end+1:-1]
    budget = max_length - 2 - len(sep_block)
    la, lb = len(ids_a), len(ids_b)
    if la + lb <= budget:
        ka, kb = la, lb
    else:
        half = budget // 2
        if la <= half:   ka, kb = la, budget - la
        elif lb <= half: ka, kb = budget - lb, lb
        else:            ka, kb = half, budget - half
    return [cls] + ids_a[:ka] + sep_block + ids_b[:kb] + [sep]


class DebertaEntailment:
    def __init__(self, model_name="microsoft/deberta-large-mnli", device=None,
                 batch_size=16, cache_dir=DEFAULT_CACHE_DIR, overlength="symmetric"):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.overlength = overlength
        self.truncated = set()
        self.n_forward = 0
        self.n_requested = 0
        self.torch = torch
        self.model_name = model_name
        self.batch_size = batch_size
        if device is None:
            if torch.cuda.is_available():    self.device = "cuda"
            elif torch.backends.mps.is_available(): self.device = "mps"
            else:                            self.device = "cpu"
        else:
            self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, torch_dtype=torch.float32)
        self.model.to(self.device).eval()
        labels = {v.lower(): int(k) for k, v in self.model.config.id2label.items()}
        self.entail_index = labels["entailment"]
        try:
            from huggingface_hub import snapshot_download
            self.revision = Path(snapshot_download(model_name, local_files_only=True)).name
        except Exception:
            self.revision = getattr(self.model.config, "_commit_hash", None) or "unknown"
        self.max_length = min(self.tokenizer.model_max_length, self.model.config.max_position_embeddings)
        self.cache = {}
        self.cache_file = None
        if cache_dir is not None:
            cache_dir = Path(cache_dir)
            cache_dir.mkdir(parents=True, exist_ok=True)
            tag = f"{model_name.replace('/', '__')}__{self.revision[:12]}__code_sep_string"
            self.cache_file = cache_dir / f"{tag}.jsonl"
            if self.cache_file.exists():
                with open(self.cache_file) as f:
                    for line in f:
                        rec = json.loads(line)
                        self.cache[rec["k"]] = rec["p"]
        print(f"{model_name} | {self.device} | cache: {len(self.cache)} entries")

    def _key(self, a, b, truncated=False):
        tag = "" if not truncated else chr(0) + "symmetric"
        return hashlib.sha256(f"{a}\x00{b}{tag}".encode()).hexdigest()

    def token_length(self, a, b):
        return len(self.tokenizer(f"{a} [SEP] {b}", truncation=False)["input_ids"])

    def encode(self, pairs, pad=True, truncate=False):
        if truncate and self.overlength == "symmetric":
            rows = [symmetric_input_ids(self.tokenizer, a, b, self.max_length) for a, b in pairs]
            width = max(len(r) for r in rows)
            pad_id = self.tokenizer.pad_token_id
            ids  = self.torch.tensor([r + [pad_id]*(width-len(r)) for r in rows])
            mask = self.torch.tensor([[1]*len(r) + [0]*(width-len(r)) for r in rows])
            return {"input_ids": ids, "attention_mask": mask}
        kw = {"return_tensors": "pt", "padding": True} if pad else {}
        kw.update({"truncation": True, "max_length": self.max_length} if truncate else {"truncation": False})
        return self.tokenizer([f"{a} [SEP] {b}" for a, b in pairs], **kw)

    def entail_batch(self, pairs):
        unique = list(dict.fromkeys(pairs))
        self.n_requested += len(unique)
        long = {(a, b) for a, b in unique if self.token_length(a, b) > self.max_length}
        self.truncated |= long
        new = []
        with self.torch.no_grad():
            for truncate in (False, True):
                todo = [p for p in unique if (p in long)==truncate and self._key(*p, truncate) not in self.cache]
                for i in range(0, len(todo), self.batch_size):
                    chunk = todo[i:i+self.batch_size]
                    enc = {k: v.to(self.device) for k, v in self.encode(chunk, truncate=truncate).items()}
                    probs = self.torch.softmax(self.model(**enc).logits.float(), dim=-1)[:, self.entail_index]
                    self.n_forward += len(chunk)
                    for (a, b), p in zip(chunk, probs.cpu().tolist()):
                        self.cache[self._key(a, b, truncate)] = p
                        new.append({"k": self._key(a, b, truncate), "p": p})
        if new and self.cache_file is not None:
            with open(self.cache_file, "a") as f:
                for rec in new:
                    f.write(json.dumps(rec) + "\n")
        return [self.cache[self._key(a, b, (a,b) in long)] for a, b in pairs]

    def describe(self):
        return {"model": self.model_name, "device": self.device}


class Similarity:
    def __init__(self, model, direction="max_bidirectional"):
        self.model = model
        self.direction = direction
        self._memo = {}

    def prefetch(self, pairs):
        pairs = [p for p in dict.fromkeys(pairs) if p not in self._memo]
        if not pairs: return
        fwd = self.model.entail_batch(pairs)
        bwd = self.model.entail_batch([(b, a) for a, b in pairs]) if self.direction == "max_bidirectional" else [None]*len(pairs)
        for p, f, b in zip(pairs, fwd, bwd):
            self._memo[p] = (f, b)

    def __call__(self, a, b):
        if (a, b) not in self._memo:
            self.prefetch([(a, b)])
        f, b_ = self._memo[(a, b)]
        return f if b_ is None else max(f, b_)

print('nli ready')

nli ready


In [4]:
# ── Dependency utilities ──────────────────────────────────────────────────────
NLI_DEP_THRESH = 0.3
USE_NLI_DEPS   = False

def get_deps(sg):
    if USE_NLI_DEPS:
        return _assign_nli_deps(sg)
    dep_map = {e["target"]: e["source"] for e in sg["edges"] if e["type"] == "data_dependency"}
    return {n["id"]: dep_map.get(n["id"]) for n in sg["nodes"] if n["type"] == "AgentStep"}

def _assign_nli_deps(sg):
    nodes = sg["nodes"]
    pairs = []
    for i, n in enumerate(nodes):
        if n["type"] != "AgentStep": continue
        srcs = nodes[:i] if n["kind"] == "tool_call" else [p for p in nodes[:i] if p["type"] == "UserTurn"]
        for s in srcs:
            pairs.append((s["result_text"], n["op_text"]))
    sim.prefetch(pairs)
    deps = {}
    for i, n in enumerate(nodes):
        if n["type"] != "AgentStep": continue
        srcs = nodes[:i] if n["kind"] == "tool_call" else [p for p in nodes[:i] if p["type"] == "UserTurn"]
        if not srcs: deps[n["id"]] = None; continue
        scores = [sim(s["result_text"], n["op_text"]) for s in srcs]
        best   = int(__import__("numpy").argmax(scores))
        deps[n["id"]] = srcs[best]["id"] if scores[best] >= NLI_DEP_THRESH else None
    return deps

def build_adj_norm(n, edge_pairs):
    import torch
    A = torch.zeros(n, n)
    for i, j in edge_pairs:
        A[i, j] = 1.0; A[j, i] = 1.0
    A = A + torch.eye(n)
    D = A.sum(1)
    D_inv_sqrt = torch.diag(D.pow(-0.5))
    return D_inv_sqrt @ A @ D_inv_sqrt

print("get_deps + build_adj_norm ready")

def get_last_user_text(nodes, before_idx):
    for n in reversed(nodes[:before_idx]):
        if n['type'] == 'UserTurn':
            return n.get('op_text') or ''
    return ''

get_deps + build_adj_norm ready


In [5]:
# ── IB Loss (used by both models) ─────────────────────────────────────────────
def ib_loss(p, mask, lam=1.0, pos_weight=5.0):
    eps = 1e-7
    H   = -(p*(p+eps).log() + (1-p)*(1-p+eps).log()).mean()
    w   = torch.where(mask > 0.5,
                      torch.ones_like(mask),
                      torch.full_like(mask, pos_weight))
    ce  = F.binary_cross_entropy(p, mask, weight=w)
    return H + lam * ce

print('IB loss ready')

IB loss ready


In [6]:
# ── gibs/mcs/mcs.py + consensus.py (inlined, essential parts) ────────────────
from dataclasses import dataclass, field
from collections import deque

ROOT = "ROOT"

@dataclass
class ProjectedGraph:
    graph_id: str
    topology: str
    nodes: dict = field(default_factory=dict)
    edges: list = field(default_factory=list)

    def out_edges(self, nid): return [e for e in self.edges if e["src"] == nid]
    def in_edges(self, nid):  return [e for e in self.edges if e["dst"] == nid]

def node_id_for(event_id): return f"N:{event_id}"

def _node_kind(step_node):
    if step_node["type"] == "UserTurn": return "user_turn"
    return "tool_result" if step_node["kind"] == "tool_call" else "agent_message"


@dataclass
class MCSConfig:
    tau_v: float = 0.7
    tau_e: float = 0.7
    top_k: int = 10
    check_source_nodes: bool = True

@dataclass
class MCSResult:
    query_id: str
    ref_id: str
    node_map: dict   # query_node_id -> ref_node_id
    edge_map: dict   # query_edge_id -> ref_edge_id
    size: int


def _node_sim(nq, nr, sim_fn):
    if nq["id"] == ROOT and nr["id"] == ROOT: return 1.0
    if nq["id"] == ROOT or nr["id"] == ROOT:  return 0.0
    if nq["text"] is None or nr["text"] is None: return 1.0
    return sim_fn(nq["text"], nr["text"])


def _edge_score(eq, er, gq, gr, sim_fn, cfg):
    e_sim = sim_fn(eq["text"] or "", er["text"] or "") if (eq["text"] and er["text"]) else 1.0
    if e_sim < cfg.tau_e: return 0.0
    t_sim = _node_sim(gq.nodes[eq["dst"]], gr.nodes[er["dst"]], sim_fn)
    if t_sim < cfg.tau_v: return 0.0
    s_sim = _node_sim(gq.nodes[eq["src"]], gr.nodes[er["src"]], sim_fn)
    if cfg.check_source_nodes and s_sim < cfg.tau_v: return 0.0
    return e_sim + t_sim + (s_sim if cfg.check_source_nodes else 0.0)


def find_mcs(gq, gr, sim_fn, cfg=None):
    if cfg is None: cfg = MCSConfig()
    q_edges = [e for e in gq.edges if e["scorable"]]
    r_edges = [e for e in gr.edges if e["scorable"]]

    scores = {}
    pairs_to_prefetch = []
    for eq in q_edges:
        for er in r_edges:
            for ta, tb in [(eq["text"] or "", er["text"] or ""),
                           (gq.nodes[eq["dst"]]["text"] or "", gr.nodes[er["dst"]]["text"] or ""),
                           (gq.nodes[eq["src"]]["text"] or "", gr.nodes[er["src"]]["text"] or "")]:
                if ta and tb: pairs_to_prefetch.extend([(ta, tb), (tb, ta)])
    sim_fn.prefetch(pairs_to_prefetch)

    for eq in q_edges:
        for er in r_edges:
            s = _edge_score(eq, er, gq, gr, sim_fn, cfg)
            if s > 0: scores[(eq["id"], er["id"])] = s

    seeds = sorted(scores, key=scores.__getitem__, reverse=True)[:cfg.top_k]

    best_node_map, best_edge_map, best_size = {}, {}, 0

    for seed in seeds:
        qe_id, re_id = seed
        node_map = {}
        edge_map = {}

        def try_add_edge(qe_id, re_id):
            eq = next(e for e in gq.edges if e["id"] == qe_id)
            er = next(e for e in gr.edges if e["id"] == re_id)
            for qn, rn in [(eq["src"], er["src"]), (eq["dst"], er["dst"])]:
                if qn in node_map and node_map[qn] != rn: return False
                if rn in {v for k, v in node_map.items() if k != qn}: return False
            if _edge_score(eq, er, gq, gr, sim_fn, cfg) == 0: return False
            node_map[eq["src"]] = er["src"]
            node_map[eq["dst"]] = er["dst"]
            edge_map[qe_id] = re_id
            return True

        if not try_add_edge(qe_id, re_id): continue
        queue = deque([qe_id])
        visited = {qe_id}
        while queue:
            cur_qe_id = queue.popleft()
            cur_eq = next(e for e in gq.edges if e["id"] == cur_qe_id)
            for neighbor_eq in gq.out_edges(cur_eq["dst"]) + gq.in_edges(cur_eq["src"]):
                if neighbor_eq["id"] in visited: continue
                visited.add(neighbor_eq["id"])
                qsrc = neighbor_eq["src"]
                qdst = neighbor_eq["dst"]
                rsrc = node_map.get(qsrc)
                rdst = node_map.get(qdst)
                candidates = []
                if rsrc and rdst:
                    candidates = [er for er in gr.edges if er["src"]==rsrc and er["dst"]==rdst]
                elif rsrc:
                    candidates = [er for er in gr.edges if er["src"]==rsrc and er["dst"] not in {v for k,v in node_map.items() if k!=qdst}]
                elif rdst:
                    candidates = [er for er in gr.edges if er["dst"]==rdst and er["src"] not in {v for k,v in node_map.items() if k!=qsrc}]
                else:
                    candidates = [er for er in gr.edges
                                  if er["src"] not in {v for k,v in node_map.items() if k!=qsrc}
                                  and er["dst"] not in {v for k,v in node_map.items() if k!=qdst}]
                for er in sorted(candidates, key=lambda e: scores.get((neighbor_eq["id"], e["id"]), 0), reverse=True)[:1]:
                    if try_add_edge(neighbor_eq["id"], er["id"]):
                        queue.append(neighbor_eq["id"])
                        break

        size = len(edge_map) + len(node_map)
        if size > best_size:
            best_size = size
            best_node_map = dict(node_map)
            best_edge_map = dict(edge_map)

    return MCSResult(query_id=gq.graph_id, ref_id=gr.graph_id,
                     node_map=best_node_map, edge_map=best_edge_map, size=best_size)


def consensus_masks(query, results):
    def pair_indicators(query, result):
        out = {}
        for nid, node in query.nodes.items():
            if not node["scorable"]: continue
            incoming = [e["id"] for e in query.in_edges(nid)]
            out[node["event"]] = {
                "node_level": nid in result.node_map,
                "context_step_level": bool(incoming) and all(e in result.edge_map for e in incoming),
            }
        return out
    per_ref = [pair_indicators(query, r) for r in results]
    masks = {}
    for event in per_ref[0]:
        masks[event] = {
            "m_node": sum(p[event]["node_level"] for p in per_ref) / len(per_ref),
            "m_ctx":  sum(p[event]["context_step_level"] for p in per_ref) / len(per_ref),
        }
    return masks

print('mcs + consensus ready')

mcs + consensus ready


In [7]:
# ── project_agentic (agentic_gibs_colab.ipynb cell 8) ──
def project_agentic(sg, deps):
    g = ProjectedGraph(graph_id=sg['graph_id'] + '_ag', topology='agentic_nli')
    g.nodes[ROOT] = {'id': ROOT, 'kind': 'root', 'text': None, 'event': None, 'scorable': False}
    prev = ROOT
    for n in sg['nodes']:
        nid  = node_id_for(n['id'])
        scor = n['type'] == 'AgentStep'
        g.nodes[nid] = {'id': nid, 'kind': _node_kind(n), 'text': n['result_text'],
                        'event': n['id'], 'scorable': scor}
        dep_src = deps.get(n['id']) if scor else None
        src     = node_id_for(dep_src) if dep_src else prev
        g.edges.append({'id': f'{src}->{nid}', 'src': src, 'dst': nid,
                        'text': n['op_text'], 'event': n['id'], 'scorable': scor,
                        'roles': ['dependency' if dep_src else 'temporal']})
        prev = nid
    return g

In [8]:
# Shared setup 2: load runs and models (same encoder and NLI conventions as reproduction)
data = json.loads(RESULTS.read_text())
info = data['info']
tasks_map = {int(t['id']): t for t in data['tasks']}
sims_by_task = {}
for r in data['simulations']:
    if (r.get('reward_info') or {}).get('reward') is not None:
        sims_by_task.setdefault(int(r['task_id']), []).append(r)
policy_text = (info.get('environment_info') or {}).get('policy', '')
with open(ANNOT_CSV) as f:
    annots = [r for r in csv.DictReader(f) if r.get('label') in
              ('correct','incorrect','tool_error') and int(r['task_id']) in EVAL_TASKS]
labels = {(r['run_id'], int(r['step_idx'])): int(r['label'] != 'correct') for r in annots}
assert labels, 'No evaluation labels found'

In [9]:
# ── MiniLM Encoder ────────────────────────────────────────────────────────────
EMB_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM = 384

emb_tok   = AutoTokenizer.from_pretrained(EMB_MODEL_NAME)
emb_model = AutoModel.from_pretrained(EMB_MODEL_NAME).to(DEVICE).eval()

@torch.no_grad()
def encode_texts(texts, batch_size=64):
    if not texts: return torch.zeros(0, EMBED_DIM)
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = emb_tok(batch, padding=True, truncation=True,
                        max_length=128, return_tensors='pt').to(DEVICE)
        m     = enc['attention_mask'].unsqueeze(-1).float()
        h     = emb_model(**enc).last_hidden_state
        vecs.append((h * m).sum(1) / m.sum(1))
    return torch.cat(vecs).cpu()

print('MiniLM encoder ready')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM encoder ready


In [10]:
import os
CACHE_DIR = DRIVE_BASE + '/nli_cache'
os.environ['STEPWISE_NLI_CACHE'] = CACHE_DIR   # kept for reference / status cell
nli = DebertaEntailment(overlength='symmetric', batch_size=32, cache_dir=CACHE_DIR)
if nli.device == 'cuda':
    nli.model = nli.model.half()
    print('fp16 enabled')
sim = Similarity(nli)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/home/svelag11/.local/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


microsoft/deberta-large-mnli | cuda | cache: 2139915 entries
fp16 enabled


In [11]:
# Model definitions: two layers, hidden=64, same linear confidence head
class GCNLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out, bias=False)
    def forward(self, x, adj):
        return F.relu(self.lin(adj @ x))

class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1, self.layer2 = GCNLayer(386,HIDDEN), GCNLayer(HIDDEN,HIDDEN)
        self.head = nn.Linear(HIDDEN,1)
    def forward(self, g):
        h = self.layer2(self.layer1(g['x'],g['adj']),g['adj'])
        return torch.sigmoid(self.head(h)).squeeze(-1)

class RelLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.self_lin = nn.Linear(d_in,d_out)
        self.rel = nn.ModuleList([nn.Linear(d_in,d_out,bias=False) for _ in range(2)])
    def forward(self,x,adjs):
        return F.relu(self.self_lin(x) + sum(w(a @ x) for w,a in zip(self.rel,adjs)))

class RGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1, self.layer2 = RelLayer(386,HIDDEN), RelLayer(HIDDEN,HIDDEN)
        self.head = nn.Linear(HIDDEN,1)
    def forward(self,g):
        h = self.layer2(self.layer1(g['x'],g['adjs']),g['adjs'])
        return torch.sigmoid(self.head(h)).squeeze(-1)

class AttentionLayer(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W = nn.Linear(d_in,HEADS*d_out,bias=False)
        self.a_src = nn.Parameter(torch.empty(1,HEADS,d_out))
        self.a_dst = nn.Parameter(torch.empty(1,HEADS,d_out))
        nn.init.xavier_normal_(self.a_src)
        nn.init.xavier_normal_(self.a_dst)
        self.d_out = d_out
    def forward(self,x,allowed):
        h = self.W(x).view(len(x),HEADS,self.d_out)
        e = F.leaky_relu((h*self.a_src).sum(-1)[:,None,:] +
                         (h*self.a_dst).sum(-1)[None,:,:], negative_slope=0.2)
        e = e.masked_fill(~allowed[:,:,None], float('-inf'))
        alpha = torch.softmax(e,dim=1)
        return F.relu(torch.einsum('ijh,jhd->ihd',alpha,h).mean(1))

class GAT(nn.Module):
    def __init__(self,complete=False):
        super().__init__()
        self.complete = complete
        self.layer1, self.layer2 = AttentionLayer(386,HIDDEN), AttentionLayer(HIDDEN,HIDDEN)
        self.head = nn.Linear(HIDDEN,1)
    def forward(self,g):
        allowed = torch.ones_like(g['allowed']) if self.complete else g['allowed']
        h = self.layer2(self.layer1(g['x'],allowed),allowed)
        return torch.sigmoid(self.head(h)).squeeze(-1)

def make_model(name):
    return {'gcn':GCN,'rgcn':RGCN,'gat_masked':lambda:GAT(False),
            'gat_complete':lambda:GAT(True)}[name]()

In [12]:
# Shared setup 3: build features once. Original node indices match annotation step_idx.
step_graphs, graphs, skipped = {}, {}, []
for tid in tqdm(TRAIN_TASKS + EVAL_TASKS, desc='Features'):
    for run in sims_by_task.get(tid,[]):
        try:
            sg = build_step_graph(run,info,tasks_map[tid])
        except ValueError as e:
            skipped.append({'run_id':run['id'],'reason':str(e)})
            continue
        agent = [(i,n) for i,n in enumerate(sg['nodes']) if n['type']=='AgentStep']
        if not agent:
            continue
        ops = [n.get('op_text') or '' for _,n in agent]
        users = [get_last_user_text(sg['nodes'],i) for i,_ in agent]
        pairs = [(o,u) for o,u in zip(ops,users) if u]
        pairs += [(o,policy_text) for o in ops if policy_text]
        sim.prefetch(pairs)
        extra = torch.tensor([[sim(o,u) if u else 0.5,
                               sim(o,policy_text) if policy_text else 0.5]
                              for o,u in zip(ops,users)],dtype=torch.float32)
        x = torch.cat([encode_texts(ops),extra],dim=1)
        pos = {n['id']:p for p,(_,n) in enumerate(agent)}
        deps = get_deps(sg)
        temporal = [(p,p+1) for p in range(len(agent)-1)]
        dependency = [(pos[deps[n['id']]],pos[n['id']]) for _,n in agent
                      if deps.get(n['id']) in pos]
        adj = build_adj_norm(len(agent),temporal+dependency)
        graphs[run['id']] = {'x':x,'adj':adj,'allowed':adj>0,
            'adjs':[build_adj_norm(len(agent),temporal),build_adj_norm(len(agent),dependency)],
            'idx':[i for i,_ in agent],'ops':ops,'task_id':tid}
        step_graphs[run['id']] = sg
EVAL_KEYS = sorted(k for k in labels if k[0] in graphs and k[1] in graphs[k[0]]['idx'])
assert EVAL_KEYS and len({labels[k] for k in EVAL_KEYS})==2
print(f'{len(graphs)} graphs; {len(EVAL_KEYS)}/{len(labels)} evaluation steps; skipped {len(skipped)} runs')
(OUT_DIR/'skipped_runs.json').write_text(json.dumps(skipped,indent=2))

Features:   0%|          | 0/24 [00:00<?, ?it/s]

480 graphs; 2857/2857 evaluation steps; skipped 0 runs


2

In [13]:
# Legacy target reuse: structural validation of every row + deterministic sampled recomputation.
# No old .pkl data or .pt checkpoints are imported. Existing NLI score cache is shared already.
LEGACY_DIR = Path(DRIVE_BASE)/'gibs_nli'
REUSE_LEGACY_TARGETS = True
LEGACY_AUDIT_VERSION = 'schema_plus_pass_fail_samples_v1'
LEGACY_AUDIT_SAMPLES_PER_TASK = 2
LEGACY_AUDIT_ATOL = 1e-5
LEGACY_AUDIT = []

def calculate_target(kind,rid,refs):
    if kind.startswith('nli'):
        ref_ops = [o for r in refs for o in graphs[r]['ops'] if o]
        if not ref_ops:
            return None
        sim.prefetch([(q,r) for q in graphs[rid]['ops'] for r in ref_ops])
        values = [max(sim(q,r) for r in ref_ops) for q in graphs[rid]['ops']]
    elif kind=='ged_loo':
        values = ged_step_targets(rid,refs)
    elif kind=='mcs_loo':
        for r in [rid]+refs:
            if r not in projected:
                projected[r] = project_agentic(step_graphs[r],get_deps(step_graphs[r]))
        matches = [find_mcs(projected[rid],projected[r],sim,MCS_CFG) for r in refs]
        masks = consensus_masks(projected[rid],matches)
        values = [masks[step_graphs[rid]['nodes'][i]['id']]['m_node'] for i in graphs[rid]['idx']]
    else:
        raise ValueError(kind)
    return values

def record_legacy_audit(record):
    LEGACY_AUDIT.append(record)
    (OUT_DIR/'legacy_cache_audit.json').write_text(json.dumps(LEGACY_AUDIT,indent=2))
    print(f"Legacy {record['kind']} task {record['task']}: {record['status']} — {record['reason']}",flush=True)

def audit_legacy_targets(kind,tid,runs,passing):
    prefixes = {'mcs_loo':'mcs_targets_task','nli_loo':'soft_targets_loo_task'}
    if not REUSE_LEGACY_TARGETS or kind not in prefixes:
        return None
    path = LEGACY_DIR/f'{prefixes[kind]}{tid}.json'
    rec = {'kind':kind,'task':tid,'path':str(path),'validation':LEGACY_AUDIT_VERSION,
           'sampled_runs':[],'all_values_recomputed':False}
    if not path.exists():
        record_legacy_audit({**rec,'status':'missing','reason':'rebuilding this task'})
        return None
    try:
        raw = json.loads(path.read_text())
        expected = {r['id'] for r in runs}
        if set(raw)!=expected:
            raise ValueError(f'run coverage differs: cached={len(raw)}, expected={len(expected)}')
        imported = {}
        for rid in sorted(expected):
            row = {int(k):float(v) for k,v in raw[rid].items()}
            idx = graphs[rid]['idx']
            if set(row)!=set(idx):
                raise ValueError(f'step indices differ for {rid}')
            values = [row[k] for k in idx]
            if not np.isfinite(values).all() or any(v<0 or v>1 for v in values):
                raise ValueError(f'invalid score range for {rid}')
            imported[rid] = torch.tensor(values,dtype=torch.float32)
        # Include a passing and failing query when available to catch self-match differences.
        ordered = sorted(expected)
        sample = []
        for pool in [[r for r in ordered if r in passing],[r for r in ordered if r not in passing],ordered]:
            for rid in pool:
                if rid not in sample:
                    sample.append(rid)
                    break
        sample = sample[:max(1,LEGACY_AUDIT_SAMPLES_PER_TASK)]
        for rid in sample:
            refs = [r for r in passing if r!=rid]
            fresh = calculate_target(kind,rid,refs)
            if fresh is None or not np.allclose(imported[rid].numpy(),fresh,atol=LEGACY_AUDIT_ATOL,rtol=0):
                raise ValueError(f'sampled target mismatch for {rid}')
            rec['sampled_runs'].append(rid)
        record_legacy_audit({**rec,'status':'reused_after_sample_audit',
            'sha256':file_digest(path),'runs':len(imported),
            'reason':'all rows structurally valid; sampled targets agree; remaining targets not recomputed'})
        return imported
    except (ValueError,TypeError,KeyError,AttributeError,OverflowError) as e:
        record_legacy_audit({**rec,'status':'rejected','reason':str(e)+'; rebuilding this task'})
        return None

In [14]:
# Shared target builder. Targets are computed ONLY on training tasks; no test references needed.
MCS_CFG = MCSConfig(tau_v=0.7,tau_e=0.7,top_k=10)
TARGET_VERSION = 'v1'
target_memory = {}
projected = {}

def targets_for(kind):
    if kind in target_memory:
        return target_memory[kind]
    signature = {'kind':kind,'data':DATA_HASH,'train':TRAIN_TASKS,'version':TARGET_VERSION,'legacy_audit':LEGACY_AUDIT_VERSION,
                 'reuse_legacy':REUSE_LEGACY_TARGETS,'audit_samples':LEGACY_AUDIT_SAMPLES_PER_TASK,
                 'audit_atol':LEGACY_AUDIT_ATOL,'source':MODEL_SOURCE_HASH,
                 'use_nli_dependencies':USE_NLI_DEPS,'dependency_threshold':NLI_DEP_THRESH,
                 'nli_model':nli.model_name,'nli_revision':nli.revision,
                 'mcs':vars(MCS_CFG),'direction':sim.direction,'overlength':nli.overlength}
    if kind == 'ged_loo':
        signature['ged'] = GED_SIGNATURE
    tag = hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:16]
    path = OUT_DIR/f'targets_{kind}_{tag}.json'
    if path.exists():
        targets = {rid:torch.tensor(v,dtype=torch.float32) for rid,v in json.loads(path.read_text()).items()}
    else:
        targets = {}
        for tid in tqdm(TRAIN_TASKS,desc=kind):
            runs = [r for r in sims_by_task.get(tid,[]) if r['id'] in graphs]
            passing = [r['id'] for r in runs if (r.get('reward_info') or {}).get('reward',0)>=0.5]
            if len(passing)<2:
                continue
            imported = audit_legacy_targets(kind,tid,runs,passing)
            if imported is not None:
                targets.update(imported)
                continue
            for run in runs:
                rid = run['id']
                refs = [r for r in passing if kind=='nli_self' or r!=rid]
                if not refs:
                    continue
                values = calculate_target(kind,rid,refs)
                if values is None:
                    continue
                targets[rid] = torch.tensor(values,dtype=torch.float32)
        path.write_text(json.dumps({r:v.tolist() for r,v in targets.items()}))
    assert targets, f'No targets for {kind}'
    target_memory[kind] = targets
    return targets

# Freeze a common training cohort for all experiments, regardless of target recipe.
TRAIN_IDS = sorted(r['id'] for tid in TRAIN_TASKS for r in sims_by_task.get(tid,[])
                   if r['id'] in graphs and
                   sum((p.get('reward_info') or {}).get('reward',0)>=0.5 and p['id'] in graphs
                       for p in sims_by_task.get(tid,[]))>=2)
assert TRAIN_IDS
print(f'Common training cohort: {len(TRAIN_IDS)} runs')

Common training cohort: 300 runs


In [15]:
# Shared runner: fixed budget, resumable checkpoints, metrics and predictions per seed
RESULT_ROWS = {}

def on_device(g):
    return {k:([a.to(DEVICE) for a in v] if k=='adjs' else v.to(DEVICE))
            for k,v in g.items() if k in ('x','adj','allowed','adjs')}

def evaluate(conf):
    assert all(k in conf for k in EVAL_KEYS)
    y = np.array([labels[k] for k in EVAL_KEYS])
    c = np.array([conf[k] for k in EVAL_KEYS])
    assert np.isfinite(c).all()
    first = {}
    for rid,i in EVAL_KEYS:
        if labels[(rid,i)]:first[rid] = min(first.get(rid,i),i)
    fe = [k for k in EVAL_KEYS if k[1]<=first.get(k[0],float('inf'))]
    fy = [labels[k] for k in fe]
    keep = np.argsort(-c,kind='stable')[:max(1,round(0.8*len(c)))]
    bins = np.minimum((c*10).astype(int),9)
    ece = sum(np.mean(bins==b)*abs(c[bins==b].mean()-(1-y[bins==b]).mean())
              for b in range(10) if (bins==b).any())
    return {'n':len(y),'errors':int(y.sum()),'AUROC':roc_auc_score(y,1-c),
            'AUCPR':average_precision_score(y,1-c),'ACC@80':float((1-y[keep]).mean()),
            'ECE':float(ece),'first_error_n':len(fe),
            'first_error_AUROC':roc_auc_score(fy,[1-conf[k] for k in fe]) if len(set(fy))==2 else float('nan')}

@torch.no_grad()
def predict(model,graph_store):
    model.eval()
    conf = {}
    for rid in sorted({k[0] for k in EVAL_KEYS}):
        values = model(on_device(graph_store[rid])).cpu().tolist()
        conf.update({(rid,i):p for i,p in zip(graphs[rid]['idx'],values)})
    return conf

def run_experiment(name,architecture,target_kind,edge_kind="heuristic"):
    graph_store = edge_graphs_for(edge_kind)
    targets = targets_for(target_kind)
    assert set(TRAIN_IDS)<=set(targets), 'Target cohort differs; inspect skipped runs'
    assert all(len(targets[r])==len(graphs[r]['idx']) for r in TRAIN_IDS)
    # Source and content hashes prevent loading checkpoints after code/data changes.
    recipe = {'name':name,'architecture':architecture,'targets':target_kind,
              'data':DATA_HASH,'train_ids':TRAIN_IDS,'epochs':EPOCHS,'lr':LR,
              'hidden':HIDDEN,'heads':HEADS,'lambda':LAM,'pos_weight':5,
              'torch':torch.__version__,'nli_revision':nli.revision,
              'encoder_revision':getattr(emb_model.config,'_commit_hash',None),
              'target_hash':hashlib.sha256(json.dumps({r:targets[r].tolist() for r in TRAIN_IDS},sort_keys=True).encode()).hexdigest(),
              'source_hash':MODEL_SOURCE_HASH,'edge_kind':edge_kind,
              'graph_hash':graph_content_hash(graph_store,TRAIN_IDS + sorted({k[0] for k in EVAL_KEYS}))}
    tag = hashlib.sha256(json.dumps(recipe,sort_keys=True).encode()).hexdigest()[:16]
    folder = OUT_DIR/f'{name}_{tag}'
    folder.mkdir(exist_ok=True)
    (folder/'recipe.json').write_text(json.dumps(recipe,indent=2))
    rows = []
    for seed in SEEDS:
        random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
        if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
        model = make_model(architecture).to(DEVICE)
        path = folder/f'seed{seed}.pt'
        t0 = time.time()
        if path.exists() and not FORCE_RETRAIN:
            model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True))
        else:
            optimizer = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.01)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
            order = list(TRAIN_IDS)
            for epoch in range(EPOCHS):
                model.train();random.shuffle(order);loss_sum = 0.0
                for rid in order:
                    optimizer.zero_grad()
                    p = model(on_device(graph_store[rid]))
                    loss = ib_loss(p,targets[rid].to(DEVICE),lam=LAM,pos_weight=5.0)
                    assert torch.isfinite(loss), f'Nonfinite loss: {name}, {rid}'
                    loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    optimizer.step();loss_sum += loss.item()
                scheduler.step()
                if (epoch+1)%10==0:
                    print(f'{name} seed={seed} epoch={epoch+1}/{EPOCHS} loss={loss_sum/len(order):.4f}',flush=True)
            torch.save(model.state_dict(),path)
        conf = predict(model,graph_store)
        row = {'experiment':name,'architecture':architecture,'target':target_kind,'edge_kind':edge_kind,'seed':seed,
               'parameters':sum(p.numel() for p in model.parameters()),**evaluate(conf)}
        rows.append(row)
        pd.DataFrame([{'run_id':r,'step_idx':i,'label_error':labels[(r,i)],'confidence':conf[(r,i)]}
                      for r,i in EVAL_KEYS]).to_csv(folder/f'predictions_seed{seed}.csv',index=False)
        print(f'{name} seed={seed}: AUROC={row["AUROC"]:.4f}, {time.time()-t0:.0f}s',flush=True)
        del model
    table = pd.DataFrame(rows)
    table.to_csv(folder/'results.csv',index=False)
    RESULT_ROWS[name] = table
    pd.concat(RESULT_ROWS.values(),ignore_index=True).to_csv(OUT_DIR/'session_results.csv',index=False)
    display(table)
    display(table[['AUROC','AUCPR','first_error_AUROC','ACC@80','ECE']].agg(['mean','std']))
    return table

In [16]:
# Additional shared helpers: build NLI adjacency only when a corresponding experiment runs.
NLI_EDGE_THRESHOLD = 0.3
edge_graph_memory = {'heuristic':graphs}

def weighted_adj(weights):
    A = weights.clone()
    A.fill_diagonal_(1.0)
    d = A.sum(1).clamp(min=1e-8).pow(-0.5)
    return d[:,None]*A*d[None,:]

def edge_graphs_for(kind):
    if kind in edge_graph_memory:
        return edge_graph_memory[kind]
    assert kind in ('nli_hard','nli_weighted')
    out = {}
    for rid,g in tqdm(graphs.items(),desc=f'Edges: {kind}'):
        sg = step_graphs[rid]
        agent = [sg['nodes'][i] for i in g['idx']]
        count = len(agent)
        # Forward entailment only: earlier result -> current operation. No max(reverse).
        positions = [(i,j) for j in range(count) for i in range(j)]
        pairs = [(agent[i]['result_text'] or '', agent[j]['op_text'] or '') for i,j in positions]
        scores = nli.entail_batch(pairs) if pairs else []
        W = torch.zeros(count,count)
        for (i,j),score in zip(positions,scores):
            W[j,i] = float(score)
        if kind=='nli_hard':
            hard = torch.zeros_like(W)
            for j in range(1,count):
                score,i = W[j,:j].max(0)
                if score >= NLI_EDGE_THRESHOLD:
                    hard[j,int(i)] = score.new_tensor(1.0)
            W = hard
        # Same undirected message-passing convention as the original reproduction.
        W = torch.maximum(W,W.T)
        T = torch.zeros_like(W)
        for i in range(count-1):T[i,i+1] = T[i+1,i] = 1.0
        A = weighted_adj(torch.maximum(T,W))
        out[rid] = {**g,'adj':A,'allowed':A>0,'adjs':[weighted_adj(T),weighted_adj(W)]}
    edge_graph_memory[kind] = out
    return out

def graph_content_hash(store,ids):
    h = hashlib.sha256()
    for rid in sorted(set(ids)):
        h.update(rid.encode())
        for t in [store[rid]['x'],store[rid]['adj'],*store[rid]['adjs']]:
            a = t.detach().cpu().contiguous().numpy()
            h.update(str(a.shape).encode());h.update(a.tobytes())
    return h.hexdigest()

In [17]:
# Additional shared helpers: GED alignment targets, prepared lazily in experiment 13.
import networkx as nx
GED_TIMEOUT = 2.0
GED_SIGNATURE = {'version':'alignment_v1','timeout':GED_TIMEOUT,'networkx':nx.__version__,
                 'node_insert_delete':1.0,'edge_insert_delete':1.0,
                 'node_substitution':'1 - mean(operation_similarity, result_similarity)',
                 'edge_substitution':'0 if same type else 1',
                 'confidence':'matched node similarity, deletion zero, mean over references'}
ged_graphs = {}
GED_LOG = []

def ged_graph(rid):
    if rid not in ged_graphs:
        sg = step_graphs[rid]
        pg = project_agentic(sg,get_deps(sg))
        g = nx.DiGraph()
        # Keep users and ROOT as context; score AgentStep nodes only.
        for nid,node in pg.nodes.items():
            event = node.get('event')
            raw = next((v for v in sg['nodes'] if v['id']==event),None)
            g.add_node(nid,kind=node['kind'],op=(raw['op_text'] or '') if raw else '',
                       result=(raw['result_text'] or '') if raw else '')
        for e in pg.edges:
            g.add_edge(e['src'],e['dst'],kind=e['roles'][0])
        ged_graphs[rid] = g
    return ged_graphs[rid]

def ged_pair_conf(rid,ref):
    stamp = {'query':rid,'ref':ref,'data':DATA_HASH,'settings':GED_SIGNATURE,
             'source':MODEL_SOURCE_HASH,'nli_revision':nli.revision}
    tag = hashlib.sha256(json.dumps(stamp,sort_keys=True).encode()).hexdigest()[:24]
    directory = OUT_DIR/'ged_pairs';directory.mkdir(exist_ok=True)
    path = directory/f'{tag}.json'
    if path.exists():
        rec = json.loads(path.read_text());GED_LOG.append(rec['diagnostics'])
        return rec['confidence']
    q,r = ged_graph(rid),ged_graph(ref)
    pairs = []
    for _,a in q.nodes(data=True):
        for _,b in r.nodes(data=True):
            if a['kind']==b['kind'] and a['kind']!='root':
                pairs += [(a[f],b[f]) for f in ('op','result') if a[f] and b[f]]
    sim.prefetch(pairs)
    def node_similarity(a,b):
        if a['kind']!=b['kind']:return 0.0
        if a['kind']=='root':return 1.0
        values = [sim(a[f],b[f]) if a[f] and b[f] else float(a[f]==b[f]) for f in ('op','result')]
        return sum(values)/len(values)
    def node_cost(a,b):
        # Disallow cross-kind alignments when deletion+insertion is cheaper.
        return 3.0 if a['kind']!=b['kind'] else 1-node_similarity(a,b)
    # Valid edit-path fallback, with ROOT forced to ROOT.
    best_nodes = [(u,None) for u in q if u!=ROOT]+[(None,v) for v in r if v!=ROOT]+[(ROOT,ROOT)]
    best_cost = float(len(q)+len(r)-2+q.number_of_edges()+r.number_of_edges())
    fallback,candidates = True,0
    started = time.time()
    for node_path,edge_path,cost in nx.optimize_edit_paths(
            q,r,node_subst_cost=node_cost,node_del_cost=lambda a:1.0,node_ins_cost=lambda a:1.0,
            edge_subst_cost=lambda a,b:float(a['kind']!=b['kind']),
            edge_del_cost=lambda a:1.0,edge_ins_cost=lambda a:1.0,
            roots=(ROOT,ROOT),timeout=GED_TIMEOUT):
        candidates += 1
        if cost <= best_cost:
            best_nodes,best_cost,fallback = node_path,float(cost),False
    mapping = {u:v for u,v in best_nodes if u is not None}
    conf = []
    for idx in graphs[rid]['idx']:
        u = node_id_for(step_graphs[rid]['nodes'][idx]['id'])
        v = mapping.get(u)
        conf.append(node_similarity(q.nodes[u],r.nodes[v]) if v is not None else 0.0)
    diagnostic = {'query':rid,'ref':ref,'best_cost':best_cost,'fallback':fallback,
                  'candidate_paths':candidates,'solver_seconds':time.time()-started}
    rec = {'confidence':conf,'diagnostics':diagnostic}
    path.write_text(json.dumps(rec));GED_LOG.append(diagnostic)
    return conf

def ged_step_targets(rid,refs):
    values = [ged_pair_conf(rid,ref) for ref in refs]
    pd.DataFrame(GED_LOG).drop_duplicates(['query','ref']).to_csv(OUT_DIR/'ged_diagnostics.csv',index=False)
    return np.mean(values,axis=0).tolist()

In [18]:
MODEL_SOURCE_HASH = '4ff143dc82b11f8b563be206f9b48580ab02e75ba38d8203d6df19a7e928d940'

## Experiments — one runnable cell per experiment
Experiment 0 is the historical self-inclusive control. Experiments 1–2 isolate target choice;
2–3 isolate typed relations; 2–4 compare GCN with masked attention;
4–5 isolate replacing fixed connectivity with complete attention.
Experiments 6–8 repeat architecture comparisons with NLI targets, so MCS is not a prerequisite.
All experiment cells are independent after shared setup.

In [19]:
# Experiment 00_gcn_nli_self: Historical control: self-inclusive NLI targets; expected recipe mean ≈0.8739, not a guaranteed rerun value.
result_00_gcn_nli_self = run_experiment('00_gcn_nli_self', 'gcn', 'nli_self')

nli_self:   0%|          | 0/15 [00:00<?, ?it/s]

00_gcn_nli_self seed=0 epoch=10/50 loss=1.2753
00_gcn_nli_self seed=0 epoch=20/50 loss=1.1971
00_gcn_nli_self seed=0 epoch=30/50 loss=1.1516
00_gcn_nli_self seed=0 epoch=40/50 loss=1.1304
00_gcn_nli_self seed=0 epoch=50/50 loss=1.1239
00_gcn_nli_self seed=0: AUROC=0.8755, 27s
00_gcn_nli_self seed=1 epoch=10/50 loss=1.2925
00_gcn_nli_self seed=1 epoch=20/50 loss=1.2242
00_gcn_nli_self seed=1 epoch=30/50 loss=1.1621
00_gcn_nli_self seed=1 epoch=40/50 loss=1.1456
00_gcn_nli_self seed=1 epoch=50/50 loss=1.1356
00_gcn_nli_self seed=1: AUROC=0.8683, 27s
00_gcn_nli_self seed=2 epoch=10/50 loss=1.2812
00_gcn_nli_self seed=2 epoch=20/50 loss=1.1878
00_gcn_nli_self seed=2 epoch=30/50 loss=1.1431
00_gcn_nli_self seed=2 epoch=40/50 loss=1.1209
00_gcn_nli_self seed=2 epoch=50/50 loss=1.1133
00_gcn_nli_self seed=2: AUROC=0.8821, 27s
00_gcn_nli_self seed=3 epoch=10/50 loss=1.2877
00_gcn_nli_self seed=3 epoch=20/50 loss=1.2042
00_gcn_nli_self seed=3 epoch=30/50 loss=1.1504
00_gcn_nli_self seed=3 epoch

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,00_gcn_nli_self,gcn,nli_self,heuristic,0,28865,2857,210,0.875489,0.492380,0.976815,0.017545,2547,0.807140
1,00_gcn_nli_self,gcn,nli_self,heuristic,1,28865,2857,210,0.868308,0.469033,0.975941,0.022305,2547,0.807215
2,00_gcn_nli_self,gcn,nli_self,heuristic,2,28865,2857,210,0.882113,0.517149,0.977690,0.014745,2547,0.821766
3,00_gcn_nli_self,gcn,nli_self,heuristic,3,28865,2857,210,0.868667,0.455138,0.975941,0.021022,2547,0.802428
4,00_gcn_nli_self,gcn,nli_self,heuristic,4,28865,2857,210,0.876344,0.499476,0.975941,0.017425,2547,0.817152


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.874184,0.486635,0.811140,0.976465,0.018609
std,0.005792,0.024652,0.008006,0.000783,0.003039


In [20]:
# Experiment 01_gcn_nli_loo: Baseline: query excluded; historical reproduction ≈0.8597.
result_01_gcn_nli_loo = run_experiment('01_gcn_nli_loo', 'gcn', 'nli_loo')

nli_loo:   0%|          | 0/15 [00:00<?, ?it/s]

Legacy nli_loo task 16: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy nli_loo task 17: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy nli_loo task 20: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy nli_loo task 21: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy nli_loo task 22: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy nli_loo task 23: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy nli_loo task 24: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy nli_loo task 32: reused_after_samp

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,01_gcn_nli_loo,gcn,nli_loo,heuristic,0,28865,2857,210,0.855628,0.421956,0.970691,0.033798,2547,0.774820
1,01_gcn_nli_loo,gcn,nli_loo,heuristic,1,28865,2857,210,0.859665,0.458701,0.971129,0.039096,2547,0.777081
2,01_gcn_nli_loo,gcn,nli_loo,heuristic,2,28865,2857,210,0.854950,0.407516,0.971129,0.033682,2547,0.780829
3,01_gcn_nli_loo,gcn,nli_loo,heuristic,3,28865,2857,210,0.859840,0.447864,0.973316,0.034742,2547,0.779659
4,01_gcn_nli_loo,gcn,nli_loo,heuristic,4,28865,2857,210,0.858215,0.412807,0.972441,0.034979,2547,0.779624


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.857660,0.429769,0.778402,0.971741,0.035259
std,0.002267,0.022408,0.002426,0.001098,0.002219


In [21]:
# Experiment 02_gcn_mcs_loo: Change only targets vs 01: MCS node inclusion frequency.
result_02_gcn_mcs_loo = run_experiment('02_gcn_mcs_loo', 'gcn', 'mcs_loo')

mcs_loo:   0%|          | 0/15 [00:00<?, ?it/s]

Legacy mcs_loo task 16: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy mcs_loo task 17: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy mcs_loo task 20: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy mcs_loo task 21: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy mcs_loo task 22: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy mcs_loo task 23: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy mcs_loo task 24: reused_after_sample_audit — all rows structurally valid; sampled targets agree; remaining targets not recomputed
Legacy mcs_loo task 32: reused_after_samp

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,02_gcn_mcs_loo,gcn,mcs_loo,heuristic,0,28865,2857,210,0.711549,0.129091,0.943570,0.571419,2547,0.646848
1,02_gcn_mcs_loo,gcn,mcs_loo,heuristic,1,28865,2857,210,0.714842,0.129107,0.943132,0.577704,2547,0.646958
2,02_gcn_mcs_loo,gcn,mcs_loo,heuristic,2,28865,2857,210,0.721005,0.132590,0.944007,0.567086,2547,0.656630
3,02_gcn_mcs_loo,gcn,mcs_loo,heuristic,3,28865,2857,210,0.701245,0.123890,0.944007,0.574498,2547,0.641681
4,02_gcn_mcs_loo,gcn,mcs_loo,heuristic,4,28865,2857,210,0.685372,0.116452,0.941382,0.581416,2547,0.629967


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.706803,0.126226,0.644417,0.943220,0.574425
std,0.013955,0.006285,0.009719,0.001089,0.005535


In [22]:
# Experiment 03_rgcn_mcs_loo: Change only architecture vs 02: preserve temporal/dependency relation types.
result_03_rgcn_mcs_loo = run_experiment('03_rgcn_mcs_loo', 'rgcn', 'mcs_loo')

03_rgcn_mcs_loo seed=0 epoch=10/50 loss=2.8755
03_rgcn_mcs_loo seed=0 epoch=20/50 loss=2.6056
03_rgcn_mcs_loo seed=0 epoch=30/50 loss=2.4294
03_rgcn_mcs_loo seed=0 epoch=40/50 loss=2.3281
03_rgcn_mcs_loo seed=0 epoch=50/50 loss=2.3028
03_rgcn_mcs_loo seed=0: AUROC=0.7667, 36s
03_rgcn_mcs_loo seed=1 epoch=10/50 loss=2.8015
03_rgcn_mcs_loo seed=1 epoch=20/50 loss=2.5144
03_rgcn_mcs_loo seed=1 epoch=30/50 loss=2.3678
03_rgcn_mcs_loo seed=1 epoch=40/50 loss=2.2733
03_rgcn_mcs_loo seed=1 epoch=50/50 loss=2.2524
03_rgcn_mcs_loo seed=1: AUROC=0.7517, 35s
03_rgcn_mcs_loo seed=2 epoch=10/50 loss=2.7981
03_rgcn_mcs_loo seed=2 epoch=20/50 loss=2.5323
03_rgcn_mcs_loo seed=2 epoch=30/50 loss=2.3614
03_rgcn_mcs_loo seed=2 epoch=40/50 loss=2.2644
03_rgcn_mcs_loo seed=2 epoch=50/50 loss=2.2472
03_rgcn_mcs_loo seed=2: AUROC=0.7943, 35s
03_rgcn_mcs_loo seed=3 epoch=10/50 loss=2.8129
03_rgcn_mcs_loo seed=3 epoch=20/50 loss=2.5344
03_rgcn_mcs_loo seed=3 epoch=30/50 loss=2.3730
03_rgcn_mcs_loo seed=3 epoch

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,03_rgcn_mcs_loo,rgcn,mcs_loo,heuristic,0,86593,2857,210,0.766706,0.263099,0.957130,0.566105,2547,0.726541
1,03_rgcn_mcs_loo,rgcn,mcs_loo,heuristic,1,86593,2857,210,0.751670,0.272623,0.957568,0.575300,2547,0.737176
2,03_rgcn_mcs_loo,rgcn,mcs_loo,heuristic,2,86593,2857,210,0.794272,0.358457,0.961067,0.579290,2547,0.755049
3,03_rgcn_mcs_loo,rgcn,mcs_loo,heuristic,3,86593,2857,210,0.781974,0.320352,0.959318,0.563748,2547,0.743122
4,03_rgcn_mcs_loo,rgcn,mcs_loo,heuristic,4,86593,2857,210,0.801383,0.291864,0.959755,0.576243,2547,0.770016


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.779201,0.301279,0.746381,0.958968,0.572137
std,0.020254,0.038732,0.016753,0.001619,0.006797


In [23]:
# Experiment 04_gat_masked_mcs_loo: Change architecture vs 02: attention over the SAME merged heuristic graph.
result_04_gat_masked_mcs_loo = run_experiment('04_gat_masked_mcs_loo', 'gat_masked', 'mcs_loo')

04_gat_masked_mcs_loo seed=0 epoch=10/50 loss=2.9789
04_gat_masked_mcs_loo seed=0 epoch=20/50 loss=2.8133
04_gat_masked_mcs_loo seed=0 epoch=30/50 loss=2.7069
04_gat_masked_mcs_loo seed=0 epoch=40/50 loss=2.6530
04_gat_masked_mcs_loo seed=0 epoch=50/50 loss=2.6351
04_gat_masked_mcs_loo seed=0: AUROC=0.7414, 50s
04_gat_masked_mcs_loo seed=1 epoch=10/50 loss=3.0141
04_gat_masked_mcs_loo seed=1 epoch=20/50 loss=2.8105
04_gat_masked_mcs_loo seed=1 epoch=30/50 loss=2.7012
04_gat_masked_mcs_loo seed=1 epoch=40/50 loss=2.6369
04_gat_masked_mcs_loo seed=1 epoch=50/50 loss=2.6304
04_gat_masked_mcs_loo seed=1: AUROC=0.7743, 50s
04_gat_masked_mcs_loo seed=2 epoch=10/50 loss=2.9856
04_gat_masked_mcs_loo seed=2 epoch=20/50 loss=2.8199
04_gat_masked_mcs_loo seed=2 epoch=30/50 loss=2.7207
04_gat_masked_mcs_loo seed=2 epoch=40/50 loss=2.6667
04_gat_masked_mcs_loo seed=2 epoch=50/50 loss=2.6494
04_gat_masked_mcs_loo seed=2: AUROC=0.8017, 50s
04_gat_masked_mcs_loo seed=3 epoch=10/50 loss=2.9964
04_gat_m

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,04_gat_masked_mcs_loo,gat_masked,mcs_loo,heuristic,0,116289,2857,210,0.741438,0.167919,0.949256,0.593952,2547,0.656503
1,04_gat_masked_mcs_loo,gat_masked,mcs_loo,heuristic,1,116289,2857,210,0.774266,0.181634,0.953631,0.603463,2547,0.697434
2,04_gat_masked_mcs_loo,gat_masked,mcs_loo,heuristic,2,116289,2857,210,0.801673,0.280814,0.958443,0.608501,2547,0.702861
3,04_gat_masked_mcs_loo,gat_masked,mcs_loo,heuristic,3,116289,2857,210,0.790338,0.204948,0.958005,0.602629,2547,0.696234
4,04_gat_masked_mcs_loo,gat_masked,mcs_loo,heuristic,4,116289,2857,210,0.780370,0.197389,0.954943,0.593658,2547,0.696574


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.777617,0.206541,0.689921,0.954856,0.600440
std,0.022739,0.043917,0.018873,0.003730,0.006462


In [24]:
# Experiment 05_gat_complete_mcs_loo: Change only attention connectivity vs 04: every pair; targets still use heuristic MCS graphs.
result_05_gat_complete_mcs_loo = run_experiment('05_gat_complete_mcs_loo', 'gat_complete', 'mcs_loo')

05_gat_complete_mcs_loo seed=0 epoch=10/50 loss=3.7738
05_gat_complete_mcs_loo seed=0 epoch=20/50 loss=3.7496
05_gat_complete_mcs_loo seed=0 epoch=30/50 loss=3.7426
05_gat_complete_mcs_loo seed=0 epoch=40/50 loss=3.7341
05_gat_complete_mcs_loo seed=0 epoch=50/50 loss=3.7318
05_gat_complete_mcs_loo seed=0: AUROC=0.4521, 50s
05_gat_complete_mcs_loo seed=1 epoch=10/50 loss=3.7888
05_gat_complete_mcs_loo seed=1 epoch=20/50 loss=3.7497
05_gat_complete_mcs_loo seed=1 epoch=30/50 loss=3.7404
05_gat_complete_mcs_loo seed=1 epoch=40/50 loss=3.7307
05_gat_complete_mcs_loo seed=1 epoch=50/50 loss=3.7284
05_gat_complete_mcs_loo seed=1: AUROC=0.4568, 50s
05_gat_complete_mcs_loo seed=2 epoch=10/50 loss=3.7784
05_gat_complete_mcs_loo seed=2 epoch=20/50 loss=3.7532
05_gat_complete_mcs_loo seed=2 epoch=30/50 loss=3.7363
05_gat_complete_mcs_loo seed=2 epoch=40/50 loss=3.7262
05_gat_complete_mcs_loo seed=2 epoch=50/50 loss=3.7237
05_gat_complete_mcs_loo seed=2: AUROC=0.4764, 50s
05_gat_complete_mcs_loo s

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,05_gat_complete_mcs_loo,gat_complete,mcs_loo,heuristic,0,116289,2857,210,0.452062,0.063934,0.919073,0.744044,2547,0.467109
1,05_gat_complete_mcs_loo,gat_complete,mcs_loo,heuristic,1,116289,2857,210,0.456764,0.063854,0.918635,0.741944,2547,0.476870
2,05_gat_complete_mcs_loo,gat_complete,mcs_loo,heuristic,2,116289,2857,210,0.476366,0.069908,0.927822,0.745039,2547,0.493742
3,05_gat_complete_mcs_loo,gat_complete,mcs_loo,heuristic,3,116289,2857,210,0.455193,0.064010,0.920822,0.742004,2547,0.469015
4,05_gat_complete_mcs_loo,gat_complete,mcs_loo,heuristic,4,116289,2857,210,0.449240,0.061875,0.918198,0.750287,2547,0.473692


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.457925,0.064716,0.476086,0.92091,0.744664
std,0.010709,0.003037,0.010591,0.00399,0.003414


In [25]:
# Experiment 06_rgcn_nli_loo: NLI-target control for 03; compare with 01 for architecture effect.
result_06_rgcn_nli_loo = run_experiment('06_rgcn_nli_loo', 'rgcn', 'nli_loo')

06_rgcn_nli_loo seed=0 epoch=10/50 loss=1.2788
06_rgcn_nli_loo seed=0 epoch=20/50 loss=1.1896
06_rgcn_nli_loo seed=0 epoch=30/50 loss=1.1072
06_rgcn_nli_loo seed=0 epoch=40/50 loss=1.0850
06_rgcn_nli_loo seed=0 epoch=50/50 loss=1.0720
06_rgcn_nli_loo seed=0: AUROC=0.8355, 36s
06_rgcn_nli_loo seed=1 epoch=10/50 loss=1.2787
06_rgcn_nli_loo seed=1 epoch=20/50 loss=1.1739
06_rgcn_nli_loo seed=1 epoch=30/50 loss=1.1110
06_rgcn_nli_loo seed=1 epoch=40/50 loss=1.0881
06_rgcn_nli_loo seed=1 epoch=50/50 loss=1.0809
06_rgcn_nli_loo seed=1: AUROC=0.8270, 35s
06_rgcn_nli_loo seed=2 epoch=10/50 loss=1.2877
06_rgcn_nli_loo seed=2 epoch=20/50 loss=1.1492
06_rgcn_nli_loo seed=2 epoch=30/50 loss=1.1000
06_rgcn_nli_loo seed=2 epoch=40/50 loss=1.0754
06_rgcn_nli_loo seed=2 epoch=50/50 loss=1.0683
06_rgcn_nli_loo seed=2: AUROC=0.8218, 35s
06_rgcn_nli_loo seed=3 epoch=10/50 loss=1.2700
06_rgcn_nli_loo seed=3 epoch=20/50 loss=1.1604
06_rgcn_nli_loo seed=3 epoch=30/50 loss=1.1032
06_rgcn_nli_loo seed=3 epoch

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,06_rgcn_nli_loo,rgcn,nli_loo,heuristic,0,86593,2857,210,0.835486,0.366300,0.968504,0.027839,2547,0.783849
1,06_rgcn_nli_loo,rgcn,nli_loo,heuristic,1,86593,2857,210,0.827033,0.298467,0.968504,0.031600,2547,0.761128
2,06_rgcn_nli_loo,rgcn,nli_loo,heuristic,2,86593,2857,210,0.821813,0.350581,0.966317,0.036957,2547,0.781718
3,06_rgcn_nli_loo,rgcn,nli_loo,heuristic,3,86593,2857,210,0.820938,0.395725,0.966317,0.033855,2547,0.773736
4,06_rgcn_nli_loo,rgcn,nli_loo,heuristic,4,86593,2857,210,0.819168,0.326524,0.965004,0.028031,2547,0.803409


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.824888,0.347520,0.780768,0.966929,0.031657
std,0.006609,0.037192,0.015477,0.001534,0.003894


In [26]:
# Experiment 07_gat_masked_nli_loo: NLI-target control for 04; same fixed adjacency as 01.
result_07_gat_masked_nli_loo = run_experiment('07_gat_masked_nli_loo', 'gat_masked', 'nli_loo')

07_gat_masked_nli_loo seed=0 epoch=10/50 loss=1.4360
07_gat_masked_nli_loo seed=0 epoch=20/50 loss=1.3698
07_gat_masked_nli_loo seed=0 epoch=30/50 loss=1.3040
07_gat_masked_nli_loo seed=0 epoch=40/50 loss=1.2825
07_gat_masked_nli_loo seed=0 epoch=50/50 loss=1.2723
07_gat_masked_nli_loo seed=0: AUROC=0.8414, 50s
07_gat_masked_nli_loo seed=1 epoch=10/50 loss=1.4394
07_gat_masked_nli_loo seed=1 epoch=20/50 loss=1.3331
07_gat_masked_nli_loo seed=1 epoch=30/50 loss=1.2780
07_gat_masked_nli_loo seed=1 epoch=40/50 loss=1.2498
07_gat_masked_nli_loo seed=1 epoch=50/50 loss=1.2428
07_gat_masked_nli_loo seed=1: AUROC=0.8176, 50s
07_gat_masked_nli_loo seed=2 epoch=10/50 loss=1.4552
07_gat_masked_nli_loo seed=2 epoch=20/50 loss=1.3311
07_gat_masked_nli_loo seed=2 epoch=30/50 loss=1.2765
07_gat_masked_nli_loo seed=2 epoch=40/50 loss=1.2448
07_gat_masked_nli_loo seed=2 epoch=50/50 loss=1.2405
07_gat_masked_nli_loo seed=2: AUROC=0.8281, 50s
07_gat_masked_nli_loo seed=3 epoch=10/50 loss=1.4516
07_gat_m

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,07_gat_masked_nli_loo,gat_masked,nli_loo,heuristic,0,116289,2857,210,0.841411,0.389101,0.968504,0.043623,2547,0.767876
1,07_gat_masked_nli_loo,gat_masked,nli_loo,heuristic,1,116289,2857,210,0.817608,0.356558,0.962380,0.052577,2547,0.751906
2,07_gat_masked_nli_loo,gat_masked,nli_loo,heuristic,2,116289,2857,210,0.828073,0.361548,0.962380,0.047618,2547,0.768101
3,07_gat_masked_nli_loo,gat_masked,nli_loo,heuristic,3,116289,2857,210,0.836760,0.388929,0.965879,0.046753,2547,0.786176
4,07_gat_masked_nli_loo,gat_masked,nli_loo,heuristic,4,116289,2857,210,0.817218,0.345768,0.959755,0.051061,2547,0.759813


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.828214,0.368381,0.766774,0.963780,0.048326
std,0.010961,0.019681,0.012737,0.003422,0.003558


In [27]:
# Experiment 08_gat_complete_nli_loo: No heuristic graph connectivity in supervision or forward pass; compare with 07.
result_08_gat_complete_nli_loo = run_experiment('08_gat_complete_nli_loo', 'gat_complete', 'nli_loo')

08_gat_complete_nli_loo seed=0 epoch=10/50 loss=1.6691
08_gat_complete_nli_loo seed=0 epoch=20/50 loss=1.6225
08_gat_complete_nli_loo seed=0 epoch=30/50 loss=1.5943
08_gat_complete_nli_loo seed=0 epoch=40/50 loss=1.5736
08_gat_complete_nli_loo seed=0 epoch=50/50 loss=1.5696
08_gat_complete_nli_loo seed=0: AUROC=0.5979, 50s
08_gat_complete_nli_loo seed=1 epoch=10/50 loss=1.6674
08_gat_complete_nli_loo seed=1 epoch=20/50 loss=1.6315
08_gat_complete_nli_loo seed=1 epoch=30/50 loss=1.5990
08_gat_complete_nli_loo seed=1 epoch=40/50 loss=1.5767
08_gat_complete_nli_loo seed=1 epoch=50/50 loss=1.5716
08_gat_complete_nli_loo seed=1: AUROC=0.5455, 50s
08_gat_complete_nli_loo seed=2 epoch=10/50 loss=1.6737
08_gat_complete_nli_loo seed=2 epoch=20/50 loss=1.6430
08_gat_complete_nli_loo seed=2 epoch=30/50 loss=1.6076
08_gat_complete_nli_loo seed=2 epoch=40/50 loss=1.5948
08_gat_complete_nli_loo seed=2 epoch=50/50 loss=1.5923
08_gat_complete_nli_loo seed=2: AUROC=0.5464, 50s
08_gat_complete_nli_loo s

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,08_gat_complete_nli_loo,gat_complete,nli_loo,heuristic,0,116289,2857,210,0.597857,0.098529,0.933071,0.059866,2547,0.619476
1,08_gat_complete_nli_loo,gat_complete,nli_loo,heuristic,1,116289,2857,210,0.545548,0.096559,0.926072,0.070862,2547,0.576622
2,08_gat_complete_nli_loo,gat_complete,nli_loo,heuristic,2,116289,2857,210,0.546398,0.078197,0.923885,0.063889,2547,0.582649
3,08_gat_complete_nli_loo,gat_complete,nli_loo,heuristic,3,116289,2857,210,0.581041,0.092608,0.931321,0.055652,2547,0.601332
4,08_gat_complete_nli_loo,gat_complete,nli_loo,heuristic,4,116289,2857,210,0.538595,0.094335,0.926509,0.059908,2547,0.575195


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.561888,0.092045,0.591055,0.928171,0.062035
std,0.026030,0.008058,0.018999,0.003856,0.005730


## Additional experiments: NLI edges and GED targets
09–12 hold NLI-LOO training targets fixed and change only dependency connectivity
relative to 01/06. Temporal edges remain unchanged. 13 keeps the original GCN graph
and changes only targets relative to 01/02. These cells run independently after setup.
Weighted-edge experiments use GCN/R-GCN because standard GAT learns attention weights
rather than directly consuming these precomputed edge weights.

In [28]:
# Experiment 09_gcn_nli_hard_edges: Compare with 01: replace value-match dependencies with best directional NLI parent.
result_09_gcn_nli_hard_edges = run_experiment('09_gcn_nli_hard_edges', 'gcn', 'nli_loo', edge_kind='nli_hard')

Edges: nli_hard:   0%|          | 0/480 [00:00<?, ?it/s]

09_gcn_nli_hard_edges seed=0 epoch=10/50 loss=1.4848
09_gcn_nli_hard_edges seed=0 epoch=20/50 loss=1.4275
09_gcn_nli_hard_edges seed=0 epoch=30/50 loss=1.3784
09_gcn_nli_hard_edges seed=0 epoch=40/50 loss=1.3684
09_gcn_nli_hard_edges seed=0 epoch=50/50 loss=1.3597
09_gcn_nli_hard_edges seed=0: AUROC=0.8576, 27s
09_gcn_nli_hard_edges seed=1 epoch=10/50 loss=1.4941
09_gcn_nli_hard_edges seed=1 epoch=20/50 loss=1.4378
09_gcn_nli_hard_edges seed=1 epoch=30/50 loss=1.4007
09_gcn_nli_hard_edges seed=1 epoch=40/50 loss=1.3796
09_gcn_nli_hard_edges seed=1 epoch=50/50 loss=1.3718
09_gcn_nli_hard_edges seed=1: AUROC=0.8404, 27s
09_gcn_nli_hard_edges seed=2 epoch=10/50 loss=1.5030
09_gcn_nli_hard_edges seed=2 epoch=20/50 loss=1.4174
09_gcn_nli_hard_edges seed=2 epoch=30/50 loss=1.3766
09_gcn_nli_hard_edges seed=2 epoch=40/50 loss=1.3596
09_gcn_nli_hard_edges seed=2 epoch=50/50 loss=1.3537
09_gcn_nli_hard_edges seed=2: AUROC=0.8477, 27s
09_gcn_nli_hard_edges seed=3 epoch=10/50 loss=1.5054
09_gcn_n

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,09_gcn_nli_hard_edges,gcn,nli_loo,nli_hard,0,28865,2857,210,0.857617,0.419790,0.971566,0.028535,2547,0.799389
1,09_gcn_nli_hard_edges,gcn,nli_loo,nli_hard,1,28865,2857,210,0.840366,0.391822,0.968066,0.036191,2547,0.780293
2,09_gcn_nli_hard_edges,gcn,nli_loo,nli_hard,2,28865,2857,210,0.847742,0.391262,0.968941,0.034621,2547,0.785605
3,09_gcn_nli_hard_edges,gcn,nli_loo,nli_hard,3,28865,2857,210,0.849185,0.432191,0.968941,0.037599,2547,0.789365
4,09_gcn_nli_hard_edges,gcn,nli_loo,nli_hard,4,28865,2857,210,0.850072,0.424822,0.968941,0.033070,2547,0.787964


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.848997,0.411977,0.788523,0.969291,0.034003
std,0.006155,0.019170,0.006988,0.001327,0.003495


In [29]:
# Experiment 10_gcn_nli_weighted_edges: Compare with 01/09: continuous directional NLI dependencies.
result_10_gcn_nli_weighted_edges = run_experiment('10_gcn_nli_weighted_edges', 'gcn', 'nli_loo', edge_kind='nli_weighted')

Edges: nli_weighted:   0%|          | 0/480 [00:00<?, ?it/s]

10_gcn_nli_weighted_edges seed=0 epoch=10/50 loss=1.4732
10_gcn_nli_weighted_edges seed=0 epoch=20/50 loss=1.4219
10_gcn_nli_weighted_edges seed=0 epoch=30/50 loss=1.3816
10_gcn_nli_weighted_edges seed=0 epoch=40/50 loss=1.3728
10_gcn_nli_weighted_edges seed=0 epoch=50/50 loss=1.3642
10_gcn_nli_weighted_edges seed=0: AUROC=0.8162, 27s
10_gcn_nli_weighted_edges seed=1 epoch=10/50 loss=1.4758
10_gcn_nli_weighted_edges seed=1 epoch=20/50 loss=1.4226
10_gcn_nli_weighted_edges seed=1 epoch=30/50 loss=1.3916
10_gcn_nli_weighted_edges seed=1 epoch=40/50 loss=1.3736
10_gcn_nli_weighted_edges seed=1 epoch=50/50 loss=1.3670
10_gcn_nli_weighted_edges seed=1: AUROC=0.8128, 27s
10_gcn_nli_weighted_edges seed=2 epoch=10/50 loss=1.4800
10_gcn_nli_weighted_edges seed=2 epoch=20/50 loss=1.4134
10_gcn_nli_weighted_edges seed=2 epoch=30/50 loss=1.3850
10_gcn_nli_weighted_edges seed=2 epoch=40/50 loss=1.3693
10_gcn_nli_weighted_edges seed=2 epoch=50/50 loss=1.3631
10_gcn_nli_weighted_edges seed=2: AUROC=0

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,10_gcn_nli_weighted_edges,gcn,nli_loo,nli_weighted,0,28865,2857,210,0.816202,0.345154,0.961067,0.048289,2547,0.752021
1,10_gcn_nli_weighted_edges,gcn,nli_loo,nli_weighted,1,28865,2857,210,0.812827,0.324866,0.960630,0.044185,2547,0.758423
2,10_gcn_nli_weighted_edges,gcn,nli_loo,nli_weighted,2,28865,2857,210,0.817579,0.338463,0.960630,0.044718,2547,0.751967
3,10_gcn_nli_weighted_edges,gcn,nli_loo,nli_weighted,3,28865,2857,210,0.818621,0.343375,0.959755,0.046829,2547,0.758487
4,10_gcn_nli_weighted_edges,gcn,nli_loo,nli_weighted,4,28865,2857,210,0.819296,0.362290,0.961505,0.048062,2547,0.753071


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.816905,0.342830,0.754794,0.960717,0.046416
std,0.002561,0.013474,0.003371,0.000649,0.001888


In [30]:
# Experiment 11_rgcn_nli_hard_edges: Compare with 06: separate temporal and hard-NLI relations.
result_11_rgcn_nli_hard_edges = run_experiment('11_rgcn_nli_hard_edges', 'rgcn', 'nli_loo', edge_kind='nli_hard')

11_rgcn_nli_hard_edges seed=0 epoch=10/50 loss=1.2802
11_rgcn_nli_hard_edges seed=0 epoch=20/50 loss=1.1880
11_rgcn_nli_hard_edges seed=0 epoch=30/50 loss=1.1071
11_rgcn_nli_hard_edges seed=0 epoch=40/50 loss=1.0817
11_rgcn_nli_hard_edges seed=0 epoch=50/50 loss=1.0704
11_rgcn_nli_hard_edges seed=0: AUROC=0.8062, 36s
11_rgcn_nli_hard_edges seed=1 epoch=10/50 loss=1.2742
11_rgcn_nli_hard_edges seed=1 epoch=20/50 loss=1.1651
11_rgcn_nli_hard_edges seed=1 epoch=30/50 loss=1.1021
11_rgcn_nli_hard_edges seed=1 epoch=40/50 loss=1.0751
11_rgcn_nli_hard_edges seed=1 epoch=50/50 loss=1.0669
11_rgcn_nli_hard_edges seed=1: AUROC=0.8158, 36s
11_rgcn_nli_hard_edges seed=2 epoch=10/50 loss=1.2972
11_rgcn_nli_hard_edges seed=2 epoch=20/50 loss=1.1513
11_rgcn_nli_hard_edges seed=2 epoch=30/50 loss=1.0911
11_rgcn_nli_hard_edges seed=2 epoch=40/50 loss=1.0670
11_rgcn_nli_hard_edges seed=2 epoch=50/50 loss=1.0569
11_rgcn_nli_hard_edges seed=2: AUROC=0.7924, 36s
11_rgcn_nli_hard_edges seed=3 epoch=10/50 l

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,11_rgcn_nli_hard_edges,rgcn,nli_loo,nli_hard,0,86593,2857,210,0.806228,0.341442,0.964567,0.042078,2547,0.767051
1,11_rgcn_nli_hard_edges,rgcn,nli_loo,nli_hard,1,86593,2857,210,0.815790,0.365839,0.966754,0.040017,2547,0.777075
2,11_rgcn_nli_hard_edges,rgcn,nli_loo,nli_hard,2,86593,2857,210,0.792371,0.294066,0.960192,0.047490,2547,0.761659
3,11_rgcn_nli_hard_edges,rgcn,nli_loo,nli_hard,3,86593,2857,210,0.784712,0.386389,0.961505,0.050503,2547,0.749761
4,11_rgcn_nli_hard_edges,rgcn,nli_loo,nli_hard,4,86593,2857,210,0.808961,0.338424,0.962380,0.047267,2547,0.794821


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.801612,0.345232,0.770073,0.963080,0.045471
std,0.012721,0.034631,0.016985,0.002599,0.004298


In [31]:
# Experiment 12_rgcn_nli_weighted_edges: Compare with 06/11: separate temporal and weighted-NLI relations.
result_12_rgcn_nli_weighted_edges = run_experiment('12_rgcn_nli_weighted_edges', 'rgcn', 'nli_loo', edge_kind='nli_weighted')

12_rgcn_nli_weighted_edges seed=0 epoch=10/50 loss=1.3023
12_rgcn_nli_weighted_edges seed=0 epoch=20/50 loss=1.1815
12_rgcn_nli_weighted_edges seed=0 epoch=30/50 loss=1.1070
12_rgcn_nli_weighted_edges seed=0 epoch=40/50 loss=1.0843
12_rgcn_nli_weighted_edges seed=0 epoch=50/50 loss=1.0720
12_rgcn_nli_weighted_edges seed=0: AUROC=0.7986, 36s
12_rgcn_nli_weighted_edges seed=1 epoch=10/50 loss=1.2793
12_rgcn_nli_weighted_edges seed=1 epoch=20/50 loss=1.1782
12_rgcn_nli_weighted_edges seed=1 epoch=30/50 loss=1.1090
12_rgcn_nli_weighted_edges seed=1 epoch=40/50 loss=1.0864
12_rgcn_nli_weighted_edges seed=1 epoch=50/50 loss=1.0787
12_rgcn_nli_weighted_edges seed=1: AUROC=0.7877, 36s
12_rgcn_nli_weighted_edges seed=2 epoch=10/50 loss=1.2983
12_rgcn_nli_weighted_edges seed=2 epoch=20/50 loss=1.1610
12_rgcn_nli_weighted_edges seed=2 epoch=30/50 loss=1.1039
12_rgcn_nli_weighted_edges seed=2 epoch=40/50 loss=1.0781
12_rgcn_nli_weighted_edges seed=2 epoch=50/50 loss=1.0703
12_rgcn_nli_weighted_edg

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,12_rgcn_nli_weighted_edges,rgcn,nli_loo,nli_weighted,0,86593,2857,210,0.798557,0.349170,0.964129,0.040955,2547,0.766399
1,12_rgcn_nli_weighted_edges,rgcn,nli_loo,nli_weighted,1,86593,2857,210,0.787672,0.326290,0.961067,0.058138,2547,0.727216
2,12_rgcn_nli_weighted_edges,rgcn,nli_loo,nli_weighted,2,86593,2857,210,0.797151,0.320272,0.962380,0.044124,2547,0.755583
3,12_rgcn_nli_weighted_edges,rgcn,nli_loo,nli_weighted,3,86593,2857,210,0.772573,0.316863,0.958880,0.056826,2547,0.749219
4,12_rgcn_nli_weighted_edges,rgcn,nli_loo,nli_weighted,4,86593,2857,210,0.766305,0.310082,0.958005,0.049549,2547,0.741842


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.784452,0.324535,0.748052,0.960892,0.049919
std,0.014500,0.014966,0.014726,0.002505,0.007572


In [32]:
# Experiment 13_gcn_ged_loo: Optional, slow: GED alignment targets; compare with 01 (NLI) and 02 (MCS).
result_13_gcn_ged_loo = run_experiment('13_gcn_ged_loo', 'gcn', 'ged_loo', edge_kind='heuristic')

ged_loo:   0%|          | 0/15 [00:00<?, ?it/s]

13_gcn_ged_loo seed=0 epoch=10/50 loss=3.2647
13_gcn_ged_loo seed=0 epoch=20/50 loss=3.1506
13_gcn_ged_loo seed=0 epoch=30/50 loss=3.0952
13_gcn_ged_loo seed=0 epoch=40/50 loss=3.0731
13_gcn_ged_loo seed=0 epoch=50/50 loss=3.0638
13_gcn_ged_loo seed=0: AUROC=0.7451, 27s
13_gcn_ged_loo seed=1 epoch=10/50 loss=3.2698
13_gcn_ged_loo seed=1 epoch=20/50 loss=3.1457
13_gcn_ged_loo seed=1 epoch=30/50 loss=3.1068
13_gcn_ged_loo seed=1 epoch=40/50 loss=3.0663
13_gcn_ged_loo seed=1 epoch=50/50 loss=3.0539
13_gcn_ged_loo seed=1: AUROC=0.7355, 27s
13_gcn_ged_loo seed=2 epoch=10/50 loss=3.2644
13_gcn_ged_loo seed=2 epoch=20/50 loss=3.1424
13_gcn_ged_loo seed=2 epoch=30/50 loss=3.1023
13_gcn_ged_loo seed=2 epoch=40/50 loss=3.0797
13_gcn_ged_loo seed=2 epoch=50/50 loss=3.0693
13_gcn_ged_loo seed=2: AUROC=0.7533, 27s
13_gcn_ged_loo seed=3 epoch=10/50 loss=3.2438
13_gcn_ged_loo seed=3 epoch=20/50 loss=3.1391
13_gcn_ged_loo seed=3 epoch=30/50 loss=3.0637
13_gcn_ged_loo seed=3 epoch=40/50 loss=3.0389
13_

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,13_gcn_ged_loo,gcn,ged_loo,heuristic,0,28865,2857,210,0.745109,0.231584,0.958005,0.334954,2547,0.683563
1,13_gcn_ged_loo,gcn,ged_loo,heuristic,1,28865,2857,210,0.735494,0.181544,0.954943,0.331686,2547,0.685760
2,13_gcn_ged_loo,gcn,ged_loo,heuristic,2,28865,2857,210,0.753298,0.196453,0.959318,0.333203,2547,0.704481
3,13_gcn_ged_loo,gcn,ged_loo,heuristic,3,28865,2857,210,0.736888,0.180671,0.956693,0.332511,2547,0.679428
4,13_gcn_ged_loo,gcn,ged_loo,heuristic,4,28865,2857,210,0.749073,0.184225,0.954943,0.343236,2547,0.680881


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.743973,0.194896,0.686823,0.956780,0.335118
std,0.007687,0.021465,0.010168,0.001917,0.004695


In [34]:
# Parallel R-GCN + complete-graph GAT, two layers per branch.
class RGCNGATHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.rel1 = RelLayer(386, HIDDEN)
        self.rel2 = RelLayer(HIDDEN, HIDDEN)
        self.att1 = AttentionLayer(386, HIDDEN)
        self.att2 = AttentionLayer(HIDDEN, HIDDEN)
        self.head = nn.Linear(2 * HIDDEN, 1)

    def forward(self, g):
        h_rel = self.rel2(
            self.rel1(g["x"], g["adjs"]), g["adjs"]
        )

        # Complete attention: every step can attend to every step.
        allowed = torch.ones_like(g["allowed"])
        h_att = self.att2(
            self.att1(g["x"], allowed), allowed
        )

        combined = torch.cat([h_rel, h_att], dim=1)
        return torch.sigmoid(self.head(combined)).squeeze(-1)


# Preserve the original factory; safe to rerun this cell.
if "_factory_before_hybrid" not in globals():
    _factory_before_hybrid = make_model


def make_model(name):
    if name == "rgcn_gat_hybrid":
        return RGCNGATHybrid()
    return _factory_before_hybrid(name)


print("Hybrid ready: R-GCN + complete GAT → confidence head")


Hybrid ready: R-GCN + complete GAT → confidence head


In [35]:
# Compare with experiment 03 (R-GCN) and 05 (complete GAT).
result_14 = run_experiment(
    "14_rgcn_gat_hybrid_mcs_loo_v1",
    "rgcn_gat_hybrid",
    "mcs_loo",
    edge_kind="heuristic",
)

14_rgcn_gat_hybrid_mcs_loo_v1 seed=0 epoch=10/50 loss=2.7996
14_rgcn_gat_hybrid_mcs_loo_v1 seed=0 epoch=20/50 loss=2.5541
14_rgcn_gat_hybrid_mcs_loo_v1 seed=0 epoch=30/50 loss=2.3851
14_rgcn_gat_hybrid_mcs_loo_v1 seed=0 epoch=40/50 loss=2.2987
14_rgcn_gat_hybrid_mcs_loo_v1 seed=0 epoch=50/50 loss=2.2772
14_rgcn_gat_hybrid_mcs_loo_v1 seed=0: AUROC=0.7718, 64s
14_rgcn_gat_hybrid_mcs_loo_v1 seed=1 epoch=10/50 loss=2.7981
14_rgcn_gat_hybrid_mcs_loo_v1 seed=1 epoch=20/50 loss=2.5459
14_rgcn_gat_hybrid_mcs_loo_v1 seed=1 epoch=30/50 loss=2.3798
14_rgcn_gat_hybrid_mcs_loo_v1 seed=1 epoch=40/50 loss=2.2949
14_rgcn_gat_hybrid_mcs_loo_v1 seed=1 epoch=50/50 loss=2.2729
14_rgcn_gat_hybrid_mcs_loo_v1 seed=1: AUROC=0.7493, 64s
14_rgcn_gat_hybrid_mcs_loo_v1 seed=2 epoch=10/50 loss=2.7904
14_rgcn_gat_hybrid_mcs_loo_v1 seed=2 epoch=20/50 loss=2.5284
14_rgcn_gat_hybrid_mcs_loo_v1 seed=2 epoch=30/50 loss=2.3615
14_rgcn_gat_hybrid_mcs_loo_v1 seed=2 epoch=40/50 loss=2.2579
14_rgcn_gat_hybrid_mcs_loo_v1 seed

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,14_rgcn_gat_hybrid_mcs_loo_v1,rgcn_gat_hybrid,mcs_loo,heuristic,0,202881,2857,210,0.771752,0.296725,0.956255,0.575151,2547,0.738785
1,14_rgcn_gat_hybrid_mcs_loo_v1,rgcn_gat_hybrid,mcs_loo,heuristic,1,202881,2857,210,0.749314,0.258483,0.954943,0.578592,2547,0.722977
2,14_rgcn_gat_hybrid_mcs_loo_v1,rgcn_gat_hybrid,mcs_loo,heuristic,2,202881,2857,210,0.784662,0.315005,0.959318,0.580639,2547,0.741277
3,14_rgcn_gat_hybrid_mcs_loo_v1,rgcn_gat_hybrid,mcs_loo,heuristic,3,202881,2857,210,0.759858,0.284415,0.956693,0.581310,2547,0.732897
4,14_rgcn_gat_hybrid_mcs_loo_v1,rgcn_gat_hybrid,mcs_loo,heuristic,4,202881,2857,210,0.744469,0.279077,0.955381,0.577170,2547,0.714003


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.762011,0.286741,0.729988,0.956518,0.578572
std,0.016443,0.020983,0.011379,0.001711,0.002522


In [36]:
# Compare with experiment 06 (R-GCN), 08 (complete GAT),
# and 14 to isolate the effect of target choice.
result_15 = run_experiment(
    "15_rgcn_gat_hybrid_nli_loo_v1",
    "rgcn_gat_hybrid",
    "nli_loo",
    edge_kind="heuristic",
)


15_rgcn_gat_hybrid_nli_loo_v1 seed=0 epoch=10/50 loss=1.2512
15_rgcn_gat_hybrid_nli_loo_v1 seed=0 epoch=20/50 loss=1.1574
15_rgcn_gat_hybrid_nli_loo_v1 seed=0 epoch=30/50 loss=1.0940
15_rgcn_gat_hybrid_nli_loo_v1 seed=0 epoch=40/50 loss=1.0677
15_rgcn_gat_hybrid_nli_loo_v1 seed=0 epoch=50/50 loss=1.0593
15_rgcn_gat_hybrid_nli_loo_v1 seed=0: AUROC=0.7841, 64s
15_rgcn_gat_hybrid_nli_loo_v1 seed=1 epoch=10/50 loss=1.2424
15_rgcn_gat_hybrid_nli_loo_v1 seed=1 epoch=20/50 loss=1.1367
15_rgcn_gat_hybrid_nli_loo_v1 seed=1 epoch=30/50 loss=1.0811
15_rgcn_gat_hybrid_nli_loo_v1 seed=1 epoch=40/50 loss=1.0582
15_rgcn_gat_hybrid_nli_loo_v1 seed=1 epoch=50/50 loss=1.0533
15_rgcn_gat_hybrid_nli_loo_v1 seed=1: AUROC=0.8062, 64s
15_rgcn_gat_hybrid_nli_loo_v1 seed=2 epoch=10/50 loss=1.2545
15_rgcn_gat_hybrid_nli_loo_v1 seed=2 epoch=20/50 loss=1.1329
15_rgcn_gat_hybrid_nli_loo_v1 seed=2 epoch=30/50 loss=1.0809
15_rgcn_gat_hybrid_nli_loo_v1 seed=2 epoch=40/50 loss=1.0605
15_rgcn_gat_hybrid_nli_loo_v1 seed

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,15_rgcn_gat_hybrid_nli_loo_v1,rgcn_gat_hybrid,nli_loo,heuristic,0,202881,2857,210,0.784090,0.249630,0.961505,0.044660,2547,0.718519
1,15_rgcn_gat_hybrid_nli_loo_v1,rgcn_gat_hybrid,nli_loo,heuristic,1,202881,2857,210,0.806176,0.271729,0.964129,0.040334,2547,0.767691
2,15_rgcn_gat_hybrid_nli_loo_v1,rgcn_gat_hybrid,nli_loo,heuristic,2,202881,2857,210,0.774940,0.233492,0.957568,0.047858,2547,0.730244
3,15_rgcn_gat_hybrid_nli_loo_v1,rgcn_gat_hybrid,nli_loo,heuristic,3,202881,2857,210,0.774183,0.219333,0.959318,0.047805,2547,0.727914
4,15_rgcn_gat_hybrid_nli_loo_v1,rgcn_gat_hybrid,nli_loo,heuristic,4,202881,2857,210,0.772078,0.243864,0.958005,0.053612,2547,0.735019


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.782293,0.243609,0.735878,0.960105,0.046854
std,0.014123,0.019485,0.018771,0.002721,0.004871


In [38]:

# Install POT (Python Optimal Transport) if needed.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ot") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "POT"
    ])

import ot
from scipy.sparse.csgraph import shortest_path

FGW_ALPHA = 0.5       # Fixed beforehand; do not tune on test results.
FGW_MAX_ITER = 300
FGW_VERSION = "op_nli_shortest_path_v1"

FGW_CACHE = OUT_DIR / "fgw_alignment"
FGW_CACHE.mkdir(parents=True, exist_ok=True)

_fgw_structures = {}
_fgw_targets = {}


def fgw_structure(rid):
    """Normalized shortest-path distances on the original merged graph."""
    if rid not in _fgw_structures:
        adjacency = graphs[rid]["allowed"].cpu().numpy().copy()
        np.fill_diagonal(adjacency, False)

        distances = shortest_path(
            adjacency.astype(float),
            directed=False,
            unweighted=True,
        )

        finite = distances[np.isfinite(distances)]
        diameter = float(finite.max()) if finite.size else 0.0

        # Explicit convention for disconnected nodes.
        distances[~np.isfinite(distances)] = diameter + 1.0
        distances /= max(float(distances.max()), 1.0)

        _fgw_structures[rid] = np.asarray(
            distances, dtype=np.float64
        )

    return _fgw_structures[rid]


def fgw_pair_scores(query_id, reference_id, alpha):
    """
    Align two trajectories, then score each query step by
    alignment-weighted NLI similarity to reference steps.
    """
    query_ops = graphs[query_id]["ops"]
    reference_ops = graphs[reference_id]["ops"]

    Cq = fgw_structure(query_id)
    Cr = fgw_structure(reference_id)

    signature = {
        "version": FGW_VERSION,
        "data": DATA_HASH,
        "query": query_id,
        "reference": reference_id,
        "query_ops": query_ops,
        "reference_ops": reference_ops,
        "query_structure": Cq.tolist(),
        "reference_structure": Cr.tolist(),
        "alpha": alpha,
        "max_iter": FGW_MAX_ITER,
        "pot_version": ot.__version__,
        "nli_revision": nli.revision,
        "nli_direction": sim.direction,
        "overlength": nli.overlength,
    }

    key = hashlib.sha256(
        json.dumps(signature, sort_keys=True).encode()
    ).hexdigest()
    path = FGW_CACHE / f"{key}.json"

    if path.exists():
        return np.asarray(
            json.loads(path.read_text())["scores"],
            dtype=np.float64,
        )

    pairs = [
        (query, reference)
        for query in query_ops
        for reference in reference_ops
    ]
    sim.prefetch(pairs)

    S = np.asarray(
        [sim(query, reference) for query, reference in pairs],
        dtype=np.float64,
    ).reshape(len(query_ops), len(reference_ops))

    S = np.clip(S, 0.0, 1.0)
    semantic_cost = 1.0 - S

    p = np.full(len(query_ops), 1.0 / len(query_ops))
    q = np.full(len(reference_ops), 1.0 / len(reference_ops))

    if alpha == 0.0:
        # Feature-only optimal transport control.
        T = ot.emd(p, q, semantic_cost)
        objective = float(np.sum(T * semantic_cost))
        solver_warning = None
    else:
        T, solver_log = ot.gromov.fused_gromov_wasserstein(
            semantic_cost,
            Cq,
            Cr,
            p,
            q,
            loss_fun="square_loss",
            symmetric=True,
            alpha=alpha,
            G0=np.outer(p, q),
            max_iter=FGW_MAX_ITER,
            tol_rel=1e-8,
            tol_abs=1e-8,
            log=True,
        )
        objective = float(solver_log["fgw_dist"])
        solver_warning = str(solver_log.get("warning"))

    T = np.asarray(T, dtype=np.float64)

    assert np.isfinite(T).all(), "Nonfinite transport plan"
    assert T.min() >= -1e-8, "Negative transport mass"
    assert np.allclose(T.sum(axis=1), p, atol=1e-6)
    assert np.allclose(T.sum(axis=0), q, atol=1e-6)

    # Normalize each query row into a distribution over reference steps.
    conditional_alignment = T / p[:, None]
    scores = np.clip(
        np.sum(conditional_alignment * S, axis=1),
        0.0,
        1.0,
    )

    path.write_text(json.dumps({
        "scores": scores.tolist(),
        "objective": objective,
        "solver_warning": solver_warning,
        "alpha": alpha,
        "query": query_id,
        "reference": reference_id,
    }))

    return scores


def alignment_targets(tasks, alpha):
    """Mean alignment score across other passing runs of each task."""
    cache_key = (tuple(tasks), float(alpha))

    if cache_key in _fgw_targets:
        return _fgw_targets[cache_key]

    targets = {}

    for tid in tqdm(tasks, desc=f"Alignment targets alpha={alpha}"):
        runs = [
            r for r in sims_by_task.get(tid, [])
            if r["id"] in graphs
        ]
        passing = [
            r["id"] for r in runs
            if (r.get("reward_info") or {}).get("reward", 0) >= 0.5
        ]

        for run in runs:
            rid = run["id"]
            references = [ref for ref in passing if ref != rid]

            if not references:
                continue

            per_reference = [
                fgw_pair_scores(rid, ref, alpha)
                for ref in references
            ]

            targets[rid] = torch.tensor(
                np.mean(per_reference, axis=0),
                dtype=torch.float32,
            )

    _fgw_targets[cache_key] = targets
    return targets


# Extend the existing target factory; safe to rerun this cell.
if "_targets_before_fgw" not in globals():
    _targets_before_fgw = targets_for


def targets_for(kind):
    if kind == "ot_semantic_loo":
        targets = alignment_targets(TRAIN_TASKS, alpha=0.0)
    elif kind == "fgw_loo":
        targets = alignment_targets(TRAIN_TASKS, alpha=FGW_ALPHA)
    else:
        return _targets_before_fgw(kind)

    assert set(TRAIN_IDS) <= set(targets), \
        "Some training runs lack alignment targets"

    return targets


print("Ready: semantic OT and FGW consensus targets")

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 62.6 MB/s  0:00:00m0:00:01:00:01
Ready: semantic OT and FGW consensus targets


In [39]:
# Evaluation references are used only for these training-free baselines.
# They are never used to construct the GCN training targets.
alignment_rows = []

for method, alpha in [
    ("Semantic OT anchor", 0.0),
    ("FGW anchor", FGW_ALPHA),
]:
    target_map = alignment_targets(EVAL_TASKS, alpha)

    confidence = {
        (rid, idx): float(score)
        for rid, values in target_map.items()
        for idx, score in zip(graphs[rid]["idx"], values)
    }

    missing = set(EVAL_KEYS) - set(confidence)
    assert not missing, (
        f"{method}: {len(missing)} evaluation steps lack references. "
        "Use a common evaluation subset before comparing methods."
    )

    alignment_rows.append({
        "method": method,
        **evaluate(confidence),
    })

    pd.DataFrame([
        {
            "run_id": rid,
            "step_idx": idx,
            "label_error": labels[(rid, idx)],
            "confidence": confidence[(rid, idx)],
        }
        for rid, idx in EVAL_KEYS
    ]).to_csv(
        OUT_DIR / f"alignment_anchor_alpha{alpha}.csv",
        index=False,
    )

alignment_results = pd.DataFrame(alignment_rows)
display(alignment_results)

alignment_results.to_csv(
    OUT_DIR / "alignment_anchor_results.csv",
    index=False,
)

Alignment targets alpha=0.0:   0%|          | 0/9 [00:00<?, ?it/s]

Alignment targets alpha=0.5:   0%|          | 0/9 [00:00<?, ?it/s]

,method,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,Semantic OT anchor,2857,210,0.802569,0.262807,0.959318,0.342871,2547,0.765981
1,FGW anchor,2857,210,0.817805,0.302506,0.961505,0.347183,2547,0.769782


In [40]:
# No structural term in alignment: control for experiment 17.
result_16 = run_experiment(
    "16_gcn_semantic_ot_loo_v1",
    "gcn",
    "ot_semantic_loo",
    edge_kind="heuristic",
)

Alignment targets alpha=0.0:   0%|          | 0/15 [00:00<?, ?it/s]

16_gcn_semantic_ot_loo_v1 seed=0 epoch=10/50 loss=3.4333
16_gcn_semantic_ot_loo_v1 seed=0 epoch=20/50 loss=3.3302
16_gcn_semantic_ot_loo_v1 seed=0 epoch=30/50 loss=3.2666
16_gcn_semantic_ot_loo_v1 seed=0 epoch=40/50 loss=3.2427
16_gcn_semantic_ot_loo_v1 seed=0 epoch=50/50 loss=3.2317
16_gcn_semantic_ot_loo_v1 seed=0: AUROC=0.7500, 27s
16_gcn_semantic_ot_loo_v1 seed=1 epoch=10/50 loss=3.4263
16_gcn_semantic_ot_loo_v1 seed=1 epoch=20/50 loss=3.3234
16_gcn_semantic_ot_loo_v1 seed=1 epoch=30/50 loss=3.2824
16_gcn_semantic_ot_loo_v1 seed=1 epoch=40/50 loss=3.2433
16_gcn_semantic_ot_loo_v1 seed=1 epoch=50/50 loss=3.2368
16_gcn_semantic_ot_loo_v1 seed=1: AUROC=0.7484, 27s
16_gcn_semantic_ot_loo_v1 seed=2 epoch=10/50 loss=3.4340
16_gcn_semantic_ot_loo_v1 seed=2 epoch=20/50 loss=3.3202
16_gcn_semantic_ot_loo_v1 seed=2 epoch=30/50 loss=3.2788
16_gcn_semantic_ot_loo_v1 seed=2 epoch=40/50 loss=3.2517
16_gcn_semantic_ot_loo_v1 seed=2 epoch=50/50 loss=3.2416
16_gcn_semantic_ot_loo_v1 seed=2: AUROC=0

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,16_gcn_semantic_ot_loo_v1,gcn,ot_semantic_loo,heuristic,0,28865,2857,210,0.749983,0.364842,0.955818,0.363822,2547,0.683327
1,16_gcn_semantic_ot_loo_v1,gcn,ot_semantic_loo,heuristic,1,28865,2857,210,0.748387,0.287407,0.955818,0.370128,2547,0.685864
2,16_gcn_semantic_ot_loo_v1,gcn,ot_semantic_loo,heuristic,2,28865,2857,210,0.783671,0.401586,0.959318,0.366190,2547,0.717706
3,16_gcn_semantic_ot_loo_v1,gcn,ot_semantic_loo,heuristic,3,28865,2857,210,0.753027,0.358296,0.955381,0.361335,2547,0.683344
4,16_gcn_semantic_ot_loo_v1,gcn,ot_semantic_loo,heuristic,4,28865,2857,210,0.774361,0.311383,0.958880,0.367827,2547,0.701667


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.761886,0.344703,0.694381,0.957043,0.365860
std,0.016067,0.045329,0.015114,0.001892,0.003421


In [41]:

# Same GCN, features, loss, and training settings.
# Only the target-generation method changes.
result_17 = run_experiment(
    "17_gcn_fgw_loo_v1",
    "gcn",
    "fgw_loo",
    edge_kind="heuristic",
)

Alignment targets alpha=0.5:   0%|          | 0/15 [00:00<?, ?it/s]

17_gcn_fgw_loo_v1 seed=0 epoch=10/50 loss=3.4319
17_gcn_fgw_loo_v1 seed=0 epoch=20/50 loss=3.3330
17_gcn_fgw_loo_v1 seed=0 epoch=30/50 loss=3.2630
17_gcn_fgw_loo_v1 seed=0 epoch=40/50 loss=3.2394
17_gcn_fgw_loo_v1 seed=0 epoch=50/50 loss=3.2303
17_gcn_fgw_loo_v1 seed=0: AUROC=0.7614, 27s
17_gcn_fgw_loo_v1 seed=1 epoch=10/50 loss=3.4321
17_gcn_fgw_loo_v1 seed=1 epoch=20/50 loss=3.3129
17_gcn_fgw_loo_v1 seed=1 epoch=30/50 loss=3.2609
17_gcn_fgw_loo_v1 seed=1 epoch=40/50 loss=3.2171
17_gcn_fgw_loo_v1 seed=1 epoch=50/50 loss=3.2106
17_gcn_fgw_loo_v1 seed=1: AUROC=0.7619, 27s
17_gcn_fgw_loo_v1 seed=2 epoch=10/50 loss=3.4392
17_gcn_fgw_loo_v1 seed=2 epoch=20/50 loss=3.3217
17_gcn_fgw_loo_v1 seed=2 epoch=30/50 loss=3.2734
17_gcn_fgw_loo_v1 seed=2 epoch=40/50 loss=3.2442
17_gcn_fgw_loo_v1 seed=2 epoch=50/50 loss=3.2324
17_gcn_fgw_loo_v1 seed=2: AUROC=0.7856, 27s
17_gcn_fgw_loo_v1 seed=3 epoch=10/50 loss=3.4200
17_gcn_fgw_loo_v1 seed=3 epoch=20/50 loss=3.3133
17_gcn_fgw_loo_v1 seed=3 epoch=30/5

,experiment,architecture,target,edge_kind,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,17_gcn_fgw_loo_v1,gcn,fgw_loo,heuristic,0,28865,2857,210,0.761369,0.380348,0.958005,0.364500,2547,0.698968
1,17_gcn_fgw_loo_v1,gcn,fgw_loo,heuristic,1,28865,2857,210,0.761932,0.320805,0.958005,0.370398,2547,0.683580
2,17_gcn_fgw_loo_v1,gcn,fgw_loo,heuristic,2,28865,2857,210,0.785605,0.387735,0.959318,0.371797,2547,0.722660
3,17_gcn_fgw_loo_v1,gcn,fgw_loo,heuristic,3,28865,2857,210,0.760077,0.367225,0.958443,0.364405,2547,0.693996
4,17_gcn_fgw_loo_v1,gcn,fgw_loo,heuristic,4,28865,2857,210,0.774012,0.327244,0.958880,0.381468,2547,0.698247


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.768599,0.356671,0.699490,0.95853,0.370514
std,0.011044,0.030779,0.014335,0.00057,0.006985


In [43]:

# New experiments only: existing models, targets, and runner stay unchanged.
BCE_ABLATION_VERSION = "v1"


class StepMLP(nn.Module):
    """Same step features, two hidden layers, no graph aggregation."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(386, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, 1),
        )

    def forward(self, g):
        return torch.sigmoid(
            self.network(g["x"])
        ).squeeze(-1)


def run_bce_experiment(name, architecture, target_kind):
    # Same targets, graph, cohort, and training budget as earlier experiments.
    target_map = targets_for(target_kind)
    graph_store = edge_graphs_for("heuristic")

    assert set(TRAIN_IDS) <= set(target_map)

    for rid in TRAIN_IDS:
        assert len(target_map[rid]) == len(graph_store[rid]["idx"])
        assert torch.isfinite(target_map[rid]).all()
        assert ((target_map[rid] >= 0) & (target_map[rid] <= 1)).all()

    target_hash = hashlib.sha256(
        json.dumps(
            {rid: target_map[rid].tolist() for rid in TRAIN_IDS},
            sort_keys=True,
        ).encode()
    ).hexdigest()

    recipe = {
        "version": BCE_ABLATION_VERSION,
        "experiment": name,
        "architecture": architecture,
        "target": target_kind,
        "loss": "plain_BCE",
        "data_hash": DATA_HASH,
        "target_hash": target_hash,
        "graph_hash": graph_content_hash(
            graph_store,
            TRAIN_IDS + sorted({rid for rid, _ in EVAL_KEYS}),
        ),
        "base_source": MODEL_SOURCE_HASH,
        "hidden": HIDDEN,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": 0.01,
        "gradient_clip": 1.0,
        "seeds": SEEDS,
        "torch_version": str(torch.__version__),
    }

    tag = hashlib.sha256(
        json.dumps(recipe, sort_keys=True).encode()
    ).hexdigest()[:16]

    folder = OUT_DIR / f"{name}_{tag}"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "recipe.json").write_text(json.dumps(recipe, indent=2))

    rows = []

    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = (
            StepMLP() if architecture == "mlp"
            else make_model(architecture)
        ).to(DEVICE)

        checkpoint = folder / f"seed{seed}.pt"
        started = time.time()

        if checkpoint.exists() and not FORCE_RETRAIN:
            model.load_state_dict(
                torch.load(
                    checkpoint,
                    map_location=DEVICE,
                    weights_only=True,
                )
            )
        else:
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=LR,
                weight_decay=0.01,
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=EPOCHS,
            )

            order = list(TRAIN_IDS)

            for epoch in range(EPOCHS):
                model.train()
                random.shuffle(order)
                total_loss = 0.0

                for rid in order:
                    optimizer.zero_grad()

                    prediction = model(on_device(graph_store[rid]))
                    target = target_map[rid].to(DEVICE)

                    # No entropy term and no target-dependent weighting.
                    loss = F.binary_cross_entropy(prediction, target)

                    assert torch.isfinite(loss), (
                        f"Nonfinite loss: {name}, seed={seed}, run={rid}"
                    )

                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

                    total_loss += loss.item()

                scheduler.step()

                if (epoch + 1) % 10 == 0:
                    print(
                        f"{name} seed={seed} "
                        f"epoch={epoch + 1}/{EPOCHS} "
                        f"BCE={total_loss / len(order):.4f}",
                        flush=True,
                    )

            torch.save(model.state_dict(), checkpoint)

        confidence = predict(model, graph_store)

        row = {
            "experiment": name,
            "architecture": architecture,
            "target": target_kind,
            "edge_kind": "heuristic",
            "loss": "plain_BCE",
            "seed": seed,
            "parameters": sum(p.numel() for p in model.parameters()),
            **evaluate(confidence),
        }
        rows.append(row)

        pd.DataFrame([
            {
                "run_id": rid,
                "step_idx": idx,
                "label_error": labels[(rid, idx)],
                "confidence": confidence[(rid, idx)],
            }
            for rid, idx in EVAL_KEYS
        ]).to_csv(folder / f"predictions_seed{seed}.csv", index=False)

        print(
            f"{name} seed={seed}: "
            f"AUROC={row['AUROC']:.4f}, "
            f"{time.time() - started:.0f}s",
            flush=True,
        )

        del model

    table = pd.DataFrame(rows)
    table.to_csv(folder / "results.csv", index=False)

    RESULT_ROWS[name] = table
    pd.concat(
        RESULT_ROWS.values(), ignore_index=True
    ).to_csv(OUT_DIR / "session_results.csv", index=False)

    display(table)
    display(
        table[
            ["AUROC", "AUCPR", "first_error_AUROC", "ACC@80", "ECE"]
        ].agg(["mean", "std"])
    )

    return table


print("Ready: plain-BCE loss comparisons")

Ready: plain-BCE loss comparisons


In [44]:
# Compare with experiment 17:
# same model and targets; remove entropy and target-dependent weighting.
result_18 = run_bce_experiment(
    "18_gcn_fgw_plain_bce",
    "gcn",
    "fgw_loo",
)

18_gcn_fgw_plain_bce seed=0 epoch=10/50 BCE=0.5702
18_gcn_fgw_plain_bce seed=0 epoch=20/50 BCE=0.5496
18_gcn_fgw_plain_bce seed=0 epoch=30/50 BCE=0.5386
18_gcn_fgw_plain_bce seed=0 epoch=40/50 BCE=0.5339
18_gcn_fgw_plain_bce seed=0 epoch=50/50 BCE=0.5326
18_gcn_fgw_plain_bce seed=0: AUROC=0.7794, 20s
18_gcn_fgw_plain_bce seed=1 epoch=10/50 BCE=0.5693
18_gcn_fgw_plain_bce seed=1 epoch=20/50 BCE=0.5560
18_gcn_fgw_plain_bce seed=1 epoch=30/50 BCE=0.5467
18_gcn_fgw_plain_bce seed=1 epoch=40/50 BCE=0.5408
18_gcn_fgw_plain_bce seed=1 epoch=50/50 BCE=0.5393
18_gcn_fgw_plain_bce seed=1: AUROC=0.7738, 20s
18_gcn_fgw_plain_bce seed=2 epoch=10/50 BCE=0.5656
18_gcn_fgw_plain_bce seed=2 epoch=20/50 BCE=0.5455
18_gcn_fgw_plain_bce seed=2 epoch=30/50 BCE=0.5368
18_gcn_fgw_plain_bce seed=2 epoch=40/50 BCE=0.5320
18_gcn_fgw_plain_bce seed=2 epoch=50/50 BCE=0.5307
18_gcn_fgw_plain_bce seed=2: AUROC=0.7882, 20s
18_gcn_fgw_plain_bce seed=3 epoch=10/50 BCE=0.5688
18_gcn_fgw_plain_bce seed=3 epoch=20/50 BCE

,experiment,architecture,target,edge_kind,loss,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,18_gcn_fgw_plain_bce,gcn,fgw_loo,heuristic,plain_BCE,0,28865,2857,210,0.779403,0.334026,0.961942,0.332282,2547,0.704372
1,18_gcn_fgw_plain_bce,gcn,fgw_loo,heuristic,plain_BCE,1,28865,2857,210,0.773803,0.298480,0.960192,0.332121,2547,0.685386
2,18_gcn_fgw_plain_bce,gcn,fgw_loo,heuristic,plain_BCE,2,28865,2857,210,0.788227,0.335108,0.963255,0.325657,2547,0.704695
3,18_gcn_fgw_plain_bce,gcn,fgw_loo,heuristic,plain_BCE,3,28865,2857,210,0.783572,0.340062,0.961942,0.332433,2547,0.697289
4,18_gcn_fgw_plain_bce,gcn,fgw_loo,heuristic,plain_BCE,4,28865,2857,210,0.779686,0.332861,0.960192,0.345333,2547,0.705191


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.780938,0.328107,0.699386,0.961505,0.333565
std,0.005360,0.016788,0.008473,0.001312,0.007177


In [45]:
# Compare with 18:
# does graph aggregation help beyond the 386 input features?
result_19 = run_bce_experiment(
    "19_mlp_fgw_plain_bce",
    "mlp",
    "fgw_loo",
)

19_mlp_fgw_plain_bce seed=0 epoch=10/50 BCE=0.5311
19_mlp_fgw_plain_bce seed=0 epoch=20/50 BCE=0.5127
19_mlp_fgw_plain_bce seed=0 epoch=30/50 BCE=0.5023
19_mlp_fgw_plain_bce seed=0 epoch=40/50 BCE=0.4979
19_mlp_fgw_plain_bce seed=0 epoch=50/50 BCE=0.4962
19_mlp_fgw_plain_bce seed=0: AUROC=0.7395, 20s
19_mlp_fgw_plain_bce seed=1 epoch=10/50 BCE=0.5322
19_mlp_fgw_plain_bce seed=1 epoch=20/50 BCE=0.5150
19_mlp_fgw_plain_bce seed=1 epoch=30/50 BCE=0.5054
19_mlp_fgw_plain_bce seed=1 epoch=40/50 BCE=0.4999
19_mlp_fgw_plain_bce seed=1 epoch=50/50 BCE=0.4985
19_mlp_fgw_plain_bce seed=1: AUROC=0.7575, 20s
19_mlp_fgw_plain_bce seed=2 epoch=10/50 BCE=0.5288
19_mlp_fgw_plain_bce seed=2 epoch=20/50 BCE=0.5086
19_mlp_fgw_plain_bce seed=2 epoch=30/50 BCE=0.5005
19_mlp_fgw_plain_bce seed=2 epoch=40/50 BCE=0.4957
19_mlp_fgw_plain_bce seed=2 epoch=50/50 BCE=0.4939
19_mlp_fgw_plain_bce seed=2: AUROC=0.7636, 20s
19_mlp_fgw_plain_bce seed=3 epoch=10/50 BCE=0.5298
19_mlp_fgw_plain_bce seed=3 epoch=20/50 BCE

,experiment,architecture,target,edge_kind,loss,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,19_mlp_fgw_plain_bce,mlp,fgw_loo,heuristic,plain_BCE,0,28993,2857,210,0.739524,0.325099,0.950569,0.368140,2547,0.697523
1,19_mlp_fgw_plain_bce,mlp,fgw_loo,heuristic,plain_BCE,1,28993,2857,210,0.757493,0.327666,0.952756,0.364073,2547,0.706693
2,19_mlp_fgw_plain_bce,mlp,fgw_loo,heuristic,plain_BCE,2,28993,2857,210,0.763646,0.294761,0.953193,0.362410,2547,0.716001
3,19_mlp_fgw_plain_bce,mlp,fgw_loo,heuristic,plain_BCE,3,28993,2857,210,0.745368,0.257456,0.951881,0.360893,2547,0.712853
4,19_mlp_fgw_plain_bce,mlp,fgw_loo,heuristic,plain_BCE,4,28993,2857,210,0.754267,0.352994,0.950569,0.369070,2547,0.706716


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.752059,0.311595,0.707957,0.951794,0.364917
std,0.009625,0.036636,0.007083,0.001214,0.003565


In [46]:

# Two separate relations: temporal and data dependency.
# Two R-GCN layers; same features, FGW targets, and plain BCE.
# Compare with 18 to test separate relation processing.
result_20 = run_bce_experiment(
    "20_rgcn2_fgw_plain_bce",
    "rgcn",
    "fgw_loo",
)


20_rgcn2_fgw_plain_bce seed=0 epoch=10/50 BCE=0.5048
20_rgcn2_fgw_plain_bce seed=0 epoch=20/50 BCE=0.4820
20_rgcn2_fgw_plain_bce seed=0 epoch=30/50 BCE=0.4693
20_rgcn2_fgw_plain_bce seed=1 epoch=30/50 BCE=0.4668
20_rgcn2_fgw_plain_bce seed=1 epoch=40/50 BCE=0.4606
20_rgcn2_fgw_plain_bce seed=1 epoch=50/50 BCE=0.4586
20_rgcn2_fgw_plain_bce seed=1: AUROC=0.7902, 29s
20_rgcn2_fgw_plain_bce seed=2 epoch=10/50 BCE=0.5023
20_rgcn2_fgw_plain_bce seed=2 epoch=20/50 BCE=0.4789
20_rgcn2_fgw_plain_bce seed=2 epoch=30/50 BCE=0.4671
20_rgcn2_fgw_plain_bce seed=2 epoch=40/50 BCE=0.4609
20_rgcn2_fgw_plain_bce seed=2 epoch=50/50 BCE=0.4590
20_rgcn2_fgw_plain_bce seed=2: AUROC=0.7680, 29s
20_rgcn2_fgw_plain_bce seed=3 epoch=10/50 BCE=0.5036
20_rgcn2_fgw_plain_bce seed=3 epoch=20/50 BCE=0.4820
20_rgcn2_fgw_plain_bce seed=3 epoch=30/50 BCE=0.4692
20_rgcn2_fgw_plain_bce seed=3 epoch=40/50 BCE=0.4623
20_rgcn2_fgw_plain_bce seed=3 epoch=50/50 BCE=0.4605
20_rgcn2_fgw_plain_bce seed=3: AUROC=0.7504, 29s
20_rg

,experiment,architecture,target,edge_kind,loss,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,20_rgcn2_fgw_plain_bce,rgcn,fgw_loo,heuristic,plain_BCE,0,86593,2857,210,0.775336,0.334863,0.956255,0.365681,2547,0.728819
1,20_rgcn2_fgw_plain_bce,rgcn,fgw_loo,heuristic,plain_BCE,1,86593,2857,210,0.790154,0.377054,0.960192,0.364000,2547,0.745868
2,20_rgcn2_fgw_plain_bce,rgcn,fgw_loo,heuristic,plain_BCE,2,86593,2857,210,0.767989,0.266683,0.954943,0.360824,2547,0.714666
3,20_rgcn2_fgw_plain_bce,rgcn,fgw_loo,heuristic,plain_BCE,3,86593,2857,210,0.750379,0.283212,0.954943,0.366221,2547,0.700265
4,20_rgcn2_fgw_plain_bce,rgcn,fgw_loo,heuristic,plain_BCE,4,86593,2857,210,0.759464,0.292488,0.954943,0.366861,2547,0.729806


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.768664,0.310860,0.723885,0.956255,0.364718
std,0.015213,0.044753,0.017216,0.002273,0.002422


In [47]:
# Compare with 01 to test the loss change on NLI targets.
# Compare with 18 to compare NLI vs FGW under the same loss.
result_21 = run_bce_experiment(
    "21_gcn_nli_loo_plain_bce",
    "gcn",
    "nli_loo",
)

21_gcn_nli_loo_plain_bce seed=0 epoch=10/50 BCE=0.2985
21_gcn_nli_loo_plain_bce seed=0 epoch=20/50 BCE=0.2912
21_gcn_nli_loo_plain_bce seed=0 epoch=30/50 BCE=0.2863
21_gcn_nli_loo_plain_bce seed=0 epoch=40/50 BCE=0.2837
21_gcn_nli_loo_plain_bce seed=0 epoch=50/50 BCE=0.2828
21_gcn_nli_loo_plain_bce seed=0: AUROC=0.8626, 20s
21_gcn_nli_loo_plain_bce seed=1 epoch=10/50 BCE=0.2982
21_gcn_nli_loo_plain_bce seed=1 epoch=20/50 BCE=0.2914
21_gcn_nli_loo_plain_bce seed=1 epoch=30/50 BCE=0.2871
21_gcn_nli_loo_plain_bce seed=1 epoch=40/50 BCE=0.2843
21_gcn_nli_loo_plain_bce seed=1 epoch=50/50 BCE=0.2835
21_gcn_nli_loo_plain_bce seed=1: AUROC=0.8563, 20s
21_gcn_nli_loo_plain_bce seed=2 epoch=10/50 BCE=0.2979
21_gcn_nli_loo_plain_bce seed=2 epoch=20/50 BCE=0.2895
21_gcn_nli_loo_plain_bce seed=2 epoch=30/50 BCE=0.2845
21_gcn_nli_loo_plain_bce seed=2 epoch=40/50 BCE=0.2818
21_gcn_nli_loo_plain_bce seed=2 epoch=50/50 BCE=0.2810
21_gcn_nli_loo_plain_bce seed=2: AUROC=0.8641, 20s
21_gcn_nli_loo_plain_b

,experiment,architecture,target,edge_kind,loss,seed,parameters,n,errors,AUROC,AUCPR,ACC@80,ECE,first_error_n,first_error_AUROC
0,21_gcn_nli_loo_plain_bce,gcn,nli_loo,heuristic,plain_BCE,0,28865,2857,210,0.862558,0.453089,0.974191,0.052915,2547,0.787958
1,21_gcn_nli_loo_plain_bce,gcn,nli_loo,heuristic,plain_BCE,1,28865,2857,210,0.856283,0.445291,0.972878,0.052059,2547,0.783984
2,21_gcn_nli_loo_plain_bce,gcn,nli_loo,heuristic,plain_BCE,2,28865,2857,210,0.864128,0.449569,0.973316,0.055921,2547,0.791908
3,21_gcn_nli_loo_plain_bce,gcn,nli_loo,heuristic,plain_BCE,3,28865,2857,210,0.869538,0.465757,0.974628,0.053044,2547,0.800271
4,21_gcn_nli_loo_plain_bce,gcn,nli_loo,heuristic,plain_BCE,4,28865,2857,210,0.858113,0.417736,0.973316,0.053193,2547,0.793921


,AUROC,AUCPR,first_error_AUROC,ACC@80,ECE
mean,0.862124,0.446289,0.791609,0.973666,0.053426
std,0.005229,0.017694,0.006163,0.000719,0.001462


In [48]:
# Summary of experiments run in this kernel (each experiment also has its own results.csv)
assert RESULT_ROWS, 'Run at least one experiment first'
results = pd.concat(RESULT_ROWS.values(),ignore_index=True)
assert results['n'].nunique()==1
summary = results.groupby('experiment')[['AUROC','AUCPR','first_error_AUROC','ACC@80','ECE']].agg(['mean','std'])
display(summary.round(4))
summary.to_csv(OUT_DIR/'session_summary.csv')
print('Mean ± sample standard deviation across seeds; no test-set model selection.')

AUROC           AUCPR          \
                                 mean     std    mean     std   
experiment                                                      
00_gcn_nli_self                0.8742  0.0058  0.4866  0.0247   
01_gcn_nli_loo                 0.8577  0.0023  0.4298  0.0224   
02_gcn_mcs_loo                 0.7068  0.0140  0.1262  0.0063   
03_rgcn_mcs_loo                0.7792  0.0203  0.3013  0.0387   
04_gat_masked_mcs_loo          0.7776  0.0227  0.2065  0.0439   
05_gat_complete_mcs_loo        0.4579  0.0107  0.0647  0.0030   
06_rgcn_nli_loo                0.8249  0.0066  0.3475  0.0372   
07_gat_masked_nli_loo          0.8282  0.0110  0.3684  0.0197   
08_gat_complete_nli_loo        0.5619  0.0260  0.0920  0.0081   
09_gcn_nli_hard_edges          0.8490  0.0062  0.4120  0.0192   
10_gcn_nli_weighted_edges      0.8169  0.0026  0.3428  0.0135   
11_rgcn_nli_hard_edges         0.8016  0.0127  0.3452  0.0346   
12_rgcn_nli_weighted_edges     0.7845  0.0145  0.3245  0.0150   
13_gcn_ged_loo                 0.7440  0.0077  0.1949  0.0215   
14_rgcn_gat_hybrid_mcs_loo_v1  0.7620  0.0164  0.2867  0.0210   
15_rgcn_gat_hybrid_nli_loo_v1  0.7823  0.0141  0.2436  0.0195   
16_gcn_semantic_ot_loo_v1      0.7619  0.0161  0.3447  0.0453   
17_gcn_fgw_loo_v1              0.7686  0.0110  0.3567  0.0308   
18_gcn_fgw_plain_bce           0.7809  0.0054  0.3281  0.0168   
19_mlp_fgw_plain_bce           0.7521  0.0096  0.3116  0.0366   
20_rgcn2_fgw_plain_bce         0.7687  0.0152  0.3109  0.0448   
21_gcn_nli_loo_plain_bce       0.8621  0.0052  0.4463  0.0177   

                              first_error_AUROC          ACC@80          \
                                           mean     std    mean     std   
experiment                                                                
00_gcn_nli_self                          0.8111  0.0080  0.9765  0.0008   
01_gcn_nli_loo                           0.7784  0.0024  0.9717  0.0011   
02_gcn_mcs_loo                           0.6444  0.0097  0.9432  0.0011   
03_rgcn_mcs_loo                          0.7464  0.0168  0.9590  0.0016   
04_gat_masked_mcs_loo                    0.6899  0.0189  0.9549  0.0037   
05_gat_complete_mcs_loo                  0.4761  0.0106  0.9209  0.0040   
06_rgcn_nli_loo                          0.7808  0.0155  0.9669  0.0015   
07_gat_masked_nli_loo                    0.7668  0.0127  0.9638  0.0034   
08_gat_complete_nli_loo                  0.5911  0.0190  0.9282  0.0039   
09_gcn_nli_hard_edges                    0.7885  0.0070  0.9693  0.0013   
10_gcn_nli_weighted_edges                0.7548  0.0034  0.9607  0.0006   
11_rgcn_nli_hard_edges                   0.7701  0.0170  0.9631  0.0026   
12_rgcn_nli_weighted_edges               0.7481  0.0147  0.9609  0.0025   
13_gcn_ged_loo                           0.6868  0.0102  0.9568  0.0019   
14_rgcn_gat_hybrid_mcs_loo_v1            0.7300  0.0114  0.9565  0.0017   
15_rgcn_gat_hybrid_nli_loo_v1            0.7359  0.0188  0.9601  0.0027   
16_gcn_semantic_ot_loo_v1                0.6944  0.0151  0.9570  0.0019   
17_gcn_fgw_loo_v1                        0.6995  0.0143  0.9585  0.0006   
18_gcn_fgw_plain_bce                     0.6994  0.0085  0.9615  0.0013   
19_mlp_fgw_plain_bce                     0.7080  0.0071  0.9518  0.0012   
20_rgcn2_fgw_plain_bce                   0.7239  0.0172  0.9563  0.0023   
21_gcn_nli_loo_plain_bce                 0.7916  0.0062  0.9737  0.0007   

                                  ECE          
                                 mean     std  
experiment                                     
00_gcn_nli_self                0.0186  0.0030  
01_gcn_nli_loo                 0.0353  0.0022  
02_gcn_mcs_loo                 0.5744  0.0055  
03_rgcn_mcs_loo                0.5721  0.0068  
04_gat_masked_mcs_loo          0.6004  0.0065  
05_gat_complete_mcs_loo        0.7447  0.0034  
06_rgcn_nli_loo                0.0317  0.0039  
07_gat_masked_nli_loo          0.0483  0.0036  
08

Mean ± sample standard deviation across seeds; no test-set model selection.


In [19]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

NEW_ANNOT_CSV = Path(DRIVE_BASE) / "steps_to_annotate_reviewed.csv"
assert NEW_ANNOT_CSV.is_file(), f"Missing file: {NEW_ANNOT_CSV}"

# Load reviewed labels.
reviewed = pd.read_csv(
    NEW_ANNOT_CSV,
    dtype={"run_id": str},
    keep_default_na=False,
)

assert {"run_id", "step_idx", "label"} <= set(reviewed.columns)

reviewed["label"] = reviewed["label"].astype(str).str.strip().str.lower()
reviewed = reviewed[reviewed["label"] != ""].copy()

if "task_id" in reviewed.columns:
    task_ids = pd.to_numeric(reviewed["task_id"], errors="raise")
    reviewed = reviewed[task_ids.isin(EVAL_TASKS)].copy()

unknown = set(reviewed["label"]) - {"correct", "incorrect", "tool_error"}
assert not unknown, f"Unexpected labels: {unknown}"

indices = pd.to_numeric(reviewed["step_idx"], errors="raise")
assert np.isfinite(indices).all() and (indices == indices.round()).all()
reviewed["step_idx"] = indices.astype(int)
reviewed["error"] = reviewed["label"].isin(
    ["incorrect", "tool_error"]
).astype(int)

conflicts = reviewed.groupby(["run_id", "step_idx"])["error"].nunique()
assert not (conflicts > 1).any(), "Conflicting labels for the same step"

reviewed = reviewed.drop_duplicates(["run_id", "step_idx"])
reviewed_labels = {
    (r.run_id, r.step_idx): int(r.error)
    for r in reviewed.itertuples(index=False)
}

# Load saved model predictions.
saved = []

for path in sorted(OUT_DIR.glob("*/predictions_seed*.csv")):
    prediction_table = pd.read_csv(path, dtype={"run_id": str})

    confidence = {
        (r.run_id, int(r.step_idx)): float(r.confidence)
        for r in prediction_table.itertuples(index=False)
    }

    saved.append({
        "experiment": path.parent.name,
        "seed": int(path.stem.removeprefix("predictions_seed")),
        "confidence": confidence,
    })

assert saved, f"No saved predictions under {OUT_DIR}"

# Evaluate every model on exactly the same reviewed steps.
common_keys = set(reviewed_labels)
for item in saved:
    common_keys.intersection_update(item["confidence"])

keys = sorted(common_keys)
assert keys, "No matching run_id/step_idx pairs found"

y = np.array([reviewed_labels[k] for k in keys])
assert len(np.unique(y)) == 2, "AUROC requires both label classes"

# First-error subset, recomputed using the reviewed labels.
first_error = {}
for rid, idx in keys:
    if reviewed_labels[(rid, idx)]:
        first_error[rid] = min(first_error.get(rid, idx), idx)

first_error_mask = np.array([
    idx <= first_error.get(rid, float("inf"))
    for rid, idx in keys
])
first_error_y = y[first_error_mask]

rows = []

for item in saved:
    confidence = np.array([item["confidence"][k] for k in keys])
    assert np.isfinite(confidence).all()

    error_score = 1.0 - confidence

    rows.append({
        "experiment": item["experiment"],
        "seed": item["seed"],
        "n": len(keys),
        "errors": int(y.sum()),
        "AUROC": roc_auc_score(y, error_score),
        "AUCPR": average_precision_score(y, error_score),
        "first_error_AUROC": (
            roc_auc_score(
                first_error_y,
                error_score[first_error_mask],
            )
            if len(np.unique(first_error_y)) == 2
            else np.nan
        ),
    })

reviewed_results = pd.DataFrame(rows)

reviewed_summary = reviewed_results.groupby("experiment")[
    ["AUROC", "AUCPR", "first_error_AUROC"]
].agg(["mean", "std"])

print(f"Annotations: {NEW_ANNOT_CSV}")
print(f"Reviewed steps: {len(reviewed_labels)}")
print(f"Common matched steps: {len(keys)}")
print(f"Errors: {y.sum()} | Correct: {len(y) - y.sum()}")
print(f"Unmatched/excluded: {len(reviewed_labels) - len(keys)}")

display(reviewed_summary.round(4))

# Save separately without changing original results or in-memory labels.
annotation_hash = hashlib.sha256(
    NEW_ANNOT_CSV.read_bytes()
).hexdigest()[:12]

save_dir = OUT_DIR / f"annotation_recheck_{annotation_hash}"
save_dir.mkdir(exist_ok=True)

reviewed_results.to_csv(save_dir / "results.csv", index=False)
reviewed_summary.to_csv(save_dir / "summary.csv")

print(f"Saved to: {save_dir}")


Annotations: /home/svelag11/agentic/steps_to_annotate_reviewed.csv
Reviewed steps: 2857
Common matched steps: 2857
Errors: 277 | Correct: 2580
Unmatched/excluded: 0


AUROC           AUCPR  \
                                                  mean     std    mean   
experiment                                                               
00_gcn_nli_self_f834c272eeac7fa2                0.7775  0.0040  0.2760   
01_gcn_nli_loo_db28dc0202f9fda5                 0.7818  0.0027  0.2712   
02_gcn_mcs_loo_dceaef637f272ef1                 0.6807  0.0072  0.1478   
03_rgcn_mcs_loo_b27e863c1839b0ca                0.7035  0.0164  0.1796   
04_gat_masked_mcs_loo_a8540167a9b73397          0.7109  0.0165  0.1622   
05_gat_complete_mcs_loo_6861f36ae94c354e        0.4708  0.0083  0.0858   
06_rgcn_nli_loo_aabb0cae5b5239cc                0.7894  0.0069  0.2660   
07_gat_masked_nli_loo_1053708da8e7f363          0.7714  0.0056  0.2319   
08_gat_complete_nli_loo_c705b2784ffa2ae8        0.5218  0.0122  0.0992   
09_gcn_nli_hard_edges_bb4a51a61513914c          0.7916  0.0054  0.2981   
10_gcn_nli_weighted_edges_4b90bf695be26ee2      0.7323  0.0018  0.2449   
11_rgcn_nli_hard_edges_00847a707640773e         0.7738  0.0138  0.2621   
12_rgcn_nli_weighted_edges_451f8e2040da0514     0.7628  0.0078  0.2429   
13_gcn_ged_loo_ab17416fb5372bcd                 0.6923  0.0085  0.1611   
14_rgcn_gat_hybrid_mcs_loo_v1_5f57926f6fa7299e  0.6922  0.0098  0.1712   
15_rgcn_gat_hybrid_nli_loo_v1_57e108af9294b39c  0.7462  0.0125  0.2093   
16_gcn_semantic_ot_loo_v1_1d09a44cd8d8a08f      0.6880  0.0090  0.2015   
17_gcn_fgw_loo_v1_df863bf1fd829f11              0.6929  0.0093  0.2057   
18_gcn_fgw_plain_bce_bf49de83394a54bf           0.6957  0.0030  0.2025   
19_mlp_fgw_plain_bce_fd5b5c72ebbb7213           0.6947  0.0039  0.1876   
20_rgcn2_fgw_plain_bce_561bef7963ca0a3e         0.6982  0.0127  0.1848   
21_gcn_nli_loo_plain_bce_3de52dc7fa2f595c       0.7890  0.0046  0.2947   

                                                       first_error_AUROC  \
                                                   std              mean   
experiment                                                                 
00_gcn_nli_self_f834c272eeac7fa2                0.0093            0.7969   
01_gcn_nli_loo_db28dc0202f9fda5                 0.0064            0.7892   
02_gcn_mcs_loo_dceaef637f272ef1                 0.0035            0.6863   
03_rgcn_mcs_loo_b27e863c1839b0ca                0.0135            0.7453   
04_gat_masked_mcs_loo_a8540167a9b73397          0.0131            0.7258   
05_gat_complete_mcs_loo_6861f36ae94c354e        0.0019            0.4798   
06_rgcn_nli_loo_aabb0cae5b5239cc                0.0136            0.8023   
07_gat_masked_nli_loo_1053708da8e7f363          0.0094            0.7985   
08_gat_complete_nli_loo_c705b2784ffa2ae8        0.0043            0.5533   
09_gcn_nli_hard_edges_bb4a51a61513914c          0.0064            0.7908   
10_gcn_nli_weighted_edges_4b90bf695be26ee2      0.0049            0.7278   
11_rgcn_nli_hard_edges_00847a707640773e         0.0120            0.7837   
12_rgcn_nli_weighted_edges_451f8e2040da0514     0.0117            0.7713   
13_gcn_ged_loo_ab17416fb5372bcd                 0.0074            0.7027   
14_rgcn_gat_hybrid_mcs_loo_v1_5f57926f6fa7299e  0.0082            0.7331   
15_rgcn_gat_hybrid_nli_loo_v1_57e108af9294b39c  0.0116            0.7650   
16_gcn_semantic_ot_loo_v1_1d09a44cd8d8a08f      0.0101            0.6972   
17_gcn_fgw_loo_v1_df863bf1fd829f11              0.0114            0.6990   
18_gcn_fgw_plain_bce_bf49de83394a54bf           0.0078            0.7083   
19_mlp_fgw_plain_bce_fd5b5c72ebbb7213           0.0129            0.7349   
20_rgcn2_fgw_plain_bce_561bef7963ca0a3e         0.0140            0.7238   
21_gcn_nli_loo_plain_bce_3de52dc7fa2f595c       0.0064            0.7920   

                                                        
                                                   std  
experiment                                              
00_gcn_nli_self_f834c272eeac7fa2                0.0071  
01_gcn_nli_loo_db28dc0202f9fda5                 0.0021  
02_gcn_mcs_l

Saved to: /home/svelag11/agentic/gibs_controlled_experiments/annotation_recheck_93be63098c65


In [20]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# Use the reviewed annotations without changing existing global labels.
review_path = Path(DRIVE_BASE) / "steps_to_annotate_reviewed.csv"
assert review_path.is_file(), f"Missing: {review_path}"

review_df = pd.read_csv(
    review_path,
    dtype={"run_id": str},
    keep_default_na=False,
)

required = {"task_id", "run_id", "step_idx", "label"}
assert required <= set(review_df.columns), f"Required: {required}"

review_df["label"] = review_df["label"].str.strip().str.lower()
review_df["task_id"] = pd.to_numeric(
    review_df["task_id"], errors="raise"
)

expected_tasks = {1, 2, 9, 11, 12, 15, 18, 43, 47}
review_df = review_df[
    review_df["task_id"].isin(expected_tasks)
    & review_df["label"].ne("")
].copy()

assert set(review_df["task_id"]) == expected_tasks
assert set(review_df["label"]) <= {
    "correct", "incorrect", "tool_error"
}

indices = pd.to_numeric(review_df["step_idx"], errors="raise")
assert np.isfinite(indices).all()
assert (indices == indices.round()).all()

review_df["step_idx"] = indices.astype(int)
review_df["error"] = review_df["label"].isin(
    ["incorrect", "tool_error"]
).astype(int)

conflicts = review_df.groupby(["run_id", "step_idx"])["error"].nunique()
assert not (conflicts > 1).any(), "Conflicting labels for the same step"

review_df = review_df.drop_duplicates(["run_id", "step_idx"])

paper_labels = {
    (r.run_id, int(r.step_idx)): int(r.error)
    for r in review_df.itertuples(index=False)
}
paper_keys = sorted(paper_labels)

assert paper_keys, "No reviewed evaluation labels found"

paper_hash = hashlib.sha256(
    review_path.read_bytes()
).hexdigest()[:12]

paper_out = Path(OUT_DIR) / f"released_baselines_reviewed_{paper_hash}"
paper_out.mkdir(parents=True, exist_ok=True)

PAPER_ROWS = []


def paper_metrics(confidence):
    missing = set(paper_keys) - set(confidence)
    assert not missing, (
        f"{len(missing)} reviewed steps lack predictions. "
        f"Examples: {list(missing)[:3]}"
    )

    y = np.array([paper_labels[k] for k in paper_keys])
    c = np.array([confidence[k] for k in paper_keys], dtype=float)

    assert np.isfinite(c).all(), "Nonfinite confidence scores"
    assert len(np.unique(y)) == 2, "AUROC requires both classes"

    first_error = {}
    for rid, idx in paper_keys:
        if paper_labels[(rid, idx)]:
            first_error[rid] = min(first_error.get(rid, idx), idx)

    first_mask = np.array([
        idx <= first_error.get(rid, float("inf"))
        for rid, idx in paper_keys
    ])

    return {
        "n": len(y),
        "errors": int(y.sum()),
        "AUROC": roc_auc_score(y, 1 - c),
        "AUCPR": average_precision_score(y, 1 - c),
        "first_error_AUROC": (
            roc_auc_score(y[first_mask], 1 - c[first_mask])
            if len(np.unique(y[first_mask])) == 2
            else np.nan
        ),
    }


def record_paper_result(method, confidence, seed=None):
    row = {
        "method": method,
        "seed": seed,
        **paper_metrics(confidence),
    }

    # Replace an existing result if the same method/seed is rerun.
    PAPER_ROWS[:] = [
        previous for previous in PAPER_ROWS
        if not (
            previous["method"] == method
            and previous["seed"] == seed
        )
    ]
    PAPER_ROWS.append(row)

    method_tag = hashlib.sha256(method.encode()).hexdigest()[:10]

    pd.DataFrame([
        {
            "run_id": rid,
            "step_idx": idx,
            "label_error": paper_labels[(rid, idx)],
            "confidence": float(confidence[(rid, idx)]),
        }
        for rid, idx in paper_keys
    ]).to_csv(
        paper_out / f"{method_tag}_seed{seed}.csv",
        index=False,
    )

    pd.DataFrame(PAPER_ROWS).to_csv(
        paper_out / "results.csv",
        index=False,
    )

    print(
        f"{method} | seed={seed} | "
        f"AUROC={row['AUROC']:.4f} | AUCPR={row['AUCPR']:.4f}"
    )
    return row


print(
    f"Evaluation ready: {len(paper_keys)} steps, "
    f"{sum(paper_labels.values())} errors"
)
print("Outputs:", paper_out)

Evaluation ready: 2857 steps, 277 errors
Outputs: /home/svelag11/agentic/gibs_controlled_experiments/released_baselines_reviewed_93be63098c65


In [21]:
import importlib.util
import inspect
import subprocess
import sys
import json
import hashlib
import random
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch

assert "record_paper_result" in globals(), (
    "Run the reviewed-label evaluator from the previous answer first."
)

# Required by the original model, even though its prediction path
# does not use GCN message passing.
if importlib.util.find_spec("torch_geometric") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "torch-geometric"
    ])

R_ROOT = Path(DRIVE_BASE) / "official_sca_reference"

if not R_ROOT.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "--filter=blob:none", "--sparse",
        "https://github.com/Xiao0o0o/stepwise-confidence-attribution.git",
        str(R_ROOT),
    ])
    subprocess.check_call([
        "git", "-C", str(R_ROOT), "sparse-checkout", "set", "src"
    ])

R_COMMIT = subprocess.check_output(
    ["git", "-C", str(R_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()


def r_source(filename):
    matches = list((R_ROOT / "src").rglob(filename))
    assert len(matches) == 1, (
        f"Expected one {filename}; found: {matches}"
    )
    return matches[0]


def r_load(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


r_mcs_path = r_source("find_mcs.py")
r_gibs_path = r_source("GIB_MCS_wo_mcs_feature.py")
r_nibs_path = r_source("similarity_based_baseline.py")
r_bert_path = r_source("bert_graph.py")

# Original GIBS imports find_mcs by this module name.
r_mcs_module = r_load("find_mcs", r_mcs_path)
r_gibs_module = r_load("sca_original_gibs", r_gibs_path)
r_nibs_module = r_load("sca_original_nibs", r_nibs_path)

# Allow imports used by the original embedding script.
for directory in [r_bert_path.parent, R_ROOT / "src"]:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

r_bert_module = r_load("sca_original_bert", r_bert_path)

R_DIR = OUT_DIR / f"released_code_{R_COMMIT[:12]}"
R_DIR.mkdir(exist_ok=True)

(R_DIR / "source_manifest.json").write_text(json.dumps({
    "commit": R_COMMIT,
    "files": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in [r_mcs_path, r_gibs_path, r_nibs_path, r_bert_path]
    },
    "adaptations": [
        "Agentic graph projection and dataset split",
        "Full MCS mappings; no >15-node shortcut",
        "Both all-step and failed-trajectory evaluation",
    ],
}, indent=2))

print("Original source commit:", R_COMMIT)
print("Outputs:", R_DIR)

/home/svelag11/.local/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Original source commit: 1ccc8b8522f5fefc82559bdda5bf1cc0dce5acab
Outputs: /home/svelag11/agentic/gibs_controlled_experiments/released_code_1ccc8b8522f5


In [22]:
# Call the original encoder factory directly.
r_embedding_function = r_bert_module.build_embedder(DEVICE)

r_embedding_description = (
    "original_build_embedder_bert_base_uncased_CLS_max256"
)

# Verify the original function returns a valid 768-dimensional vector.
test_vector = np.asarray(
    r_embedding_function("A short test sentence.")
)

assert test_vector.shape == (768,), test_vector.shape
assert np.isfinite(test_vector).all()

print("Original BERT encoder ready")
print("Pooling: [CLS]")
print("Maximum tokens: 256")
print("Embedding dimension:", test_vector.shape[0])
# Separate cache: do NOT reuse old NLI scores with different tokenization.
R_BERT_CACHE = R_DIR / "bert_vectors"
R_BERT_CACHE.mkdir(exist_ok=True)


def r_embed(text):
    key = hashlib.sha256(
        (R_COMMIT + "\0" + r_embedding_description + "\0" + text).encode()
    ).hexdigest()
    path = R_BERT_CACHE / f"{key}.npy"

    if path.exists():
        return np.load(path, allow_pickle=False)

    vector = r_embedding_function(text)
    np.save(path, vector, allow_pickle=False)
    return vector


# Original NIBS tokenizer, FP32 NLI model, and averaging of directions.
r_similarity = r_nibs_module.SimilarityCalculator(device=DEVICE)

R_NLI_CACHE_FILE = R_DIR / "original_nli_scores.jsonl"
r_nli_cache = {}

if R_NLI_CACHE_FILE.exists():
    with R_NLI_CACHE_FILE.open() as f:
        for line in f:
            item = json.loads(line)
            r_nli_cache[item["key"]] = item["score"]

r_original_direction = (
    r_similarity._nli_entailment_probs_one_direction
)


def r_cached_direction(pairs, batch_size=64, max_length=512):
    keys = [
        hashlib.sha256(
            json.dumps(
                ["original_pair_tokens", max_length, a, b]
            ).encode()
        ).hexdigest()
        for a, b in pairs
    ]

    missing = {}
    for key, pair in zip(keys, pairs):
        if key not in r_nli_cache:
            missing[key] = pair

    if missing:
        fresh = r_original_direction(
            list(missing.values()),
            batch_size=batch_size,
            max_length=max_length,
        )
        with R_NLI_CACHE_FILE.open("a") as f:
            for key, score in zip(missing, fresh):
                r_nli_cache[key] = float(score)
                f.write(json.dumps({
                    "key": key, "score": float(score)
                }) + "\n")

    return [r_nli_cache[key] for key in keys]


r_similarity._nli_entailment_probs_one_direction = r_cached_direction

# Original MCS uses literal "[SEP]" input and MAX of directions.
# Share NLI weights, but preserve that distinct original scoring method.
r_matcher = r_mcs_module.MaximumCommonSubgraph.__new__(
    r_mcs_module.MaximumCommonSubgraph
)
r_matcher.node_threshold = 0.7
r_matcher.edge_threshold = 0.7
r_matcher.device = torch.device(DEVICE)
r_matcher.tokenizer = r_similarity.tokenizer
r_matcher.model = r_similarity.nli_model


def r_cached_mcs_similarity(a, b):
    key = hashlib.sha256(
        json.dumps(["original_mcs_sep_max", a, b]).encode()
    ).hexdigest()

    if key not in r_nli_cache:
        score = r_mcs_module.MaximumCommonSubgraph.calculate_entailment(
            r_matcher, a, b
        )
        r_nli_cache[key] = float(score)
        with R_NLI_CACHE_FILE.open("a") as f:
            f.write(json.dumps({
                "key": key, "score": float(score)
            }) + "\n")

    return r_nli_cache[key]


r_matcher.calculate_entailment = r_cached_mcs_similarity

# Original scoring methods, without invoking its file-format loader.
r_analyzer = r_nibs_module.GraphSimilarityAnalyzer.__new__(
    r_nibs_module.GraphSimilarityAnalyzer
)
r_analyzer.sim_calculator = r_similarity
r_analyzer.nli_batch_size = 64
r_analyzer.nli_max_length = 512

print("Original BERT and NLI scoring ready")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original BERT encoder ready
Pooling: [CLS]
Maximum tokens: 256
Embedding dimension: 768
Loading NLI model: microsoft/deberta-large-mnli


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI model loaded on cuda
Original BERT and NLI scoring ready


In [23]:
r_data = {}
r_graphs = {}
r_edge_steps = {}
r_success = {}

for tid in tqdm(TRAIN_TASKS + EVAL_TASKS, desc="Original-format graphs"):
    for run in sims_by_task.get(tid, []):
        rid = run["id"]
        if rid not in step_graphs:
            continue

        sg = step_graphs[rid]
        projected_graph = project_agentic(sg, get_deps(sg))

        # Retain agent-operation edges. User turns can remain source nodes.
        selected_edges = [
            e for e in projected_graph.edges if e["scorable"]
        ]
        if not selected_edges:
            continue

        node_names = {
            ROOT: "NodeRaw",
            **{
                node_id_for(node["id"]): f"Node{k}"
                for k, node in enumerate(sg["nodes"])
            },
        }

        used_nodes = {
            endpoint
            for e in selected_edges
            for endpoint in [e["src"], e["dst"]]
        }

        texts = {}
        for node_id in used_nodes:
            texts[node_names[node_id]] = (
                "Problem/Initial State"
                if node_id == ROOT
                else projected_graph.nodes[node_id]["text"] or ""
            )

        raw_index = {
            node["id"]: k for k, node in enumerate(sg["nodes"])
        }
        edge_to_step = {}
        structure = []

        for e in selected_edges:
            idx = raw_index[e["event"]]
            edge_name = f"Edge{idx}"
            texts[edge_name] = e["text"] or ""
            edge_to_step[edge_name] = idx

            structure.append(
                f"[{node_names[e['src']]}, "
                f"{node_names[e['dst']]}, {edge_name}]"
            )

        structure = ", ".join(structure)
        embeddings = {name: r_embed(text) for name, text in texts.items()}

        r_data[rid] = {
            "embeddings": embeddings,
            "reasoning_string": texts,
            "structure_str": structure,
        }
        r_graphs[rid] = r_analyzer.build_graph(
            structure, embeddings, texts, rid
        )
        r_edge_steps[rid] = edge_to_step
        r_success[rid] = (
            (run.get("reward_info") or {}).get("reward", 0) >= 0.5
        )

r_nibs_conf = {
    name: {} for name in [
        "Released NIBS NLI-Max",
        "Released NIBS NLI-Mean",
        "Released NIBS Cos-Max",
        "Released NIBS Cos-Mean",
    ]
}

for tid in tqdm(EVAL_TASKS, desc="Original NIBS formulas"):
    run_ids = [
        run["id"] for run in sims_by_task.get(tid, [])
        if run["id"] in r_data
    ]
    passing = [rid for rid in run_ids if r_success[rid]]

    for rid in run_ids:
        references = {
            ref: r_graphs[ref]
            for ref in passing if ref != rid
        }
        assert references, f"No references for {rid}"

        graph = r_graphs[rid]
        texts = r_data[rid]["reasoning_string"]

        for u, v in graph.edges():
            edge_name = graph.edges[u, v]["label"]
            step_idx = r_edge_steps[rid][edge_name]
            edge_text = texts[edge_name]
            result_text = texts[v]
            embedding = graph.edges[u, v]["embedding"]

            values = {
                "Released NIBS NLI-Max": r_analyzer.nli_max(
                    edge_text, result_text, references
                ),
                "Released NIBS NLI-Mean": r_analyzer.nli_mean(
                    edge_text, result_text, references
                ),
                "Released NIBS Cos-Max": r_analyzer.cosine_max(
                    embedding, references
                ),
                "Released NIBS Cos-Mean": r_analyzer.cosine_mean(
                    embedding, references
                ),
            }

            for method, value in values.items():
                r_nibs_conf[method][(rid, step_idx)] = float(value)

for method, confidence in r_nibs_conf.items():
    record_paper_result(method, confidence)

print("Original formulas evaluated on ALL reviewed evaluation steps.")

Original-format graphs:   0%|          | 0/24 [00:00<?, ?it/s]

Original NIBS formulas:   0%|          | 0/9 [00:00<?, ?it/s]

Released NIBS NLI-Max | seed=None | AUROC=0.7220 | AUCPR=0.2018
Released NIBS NLI-Mean | seed=None | AUROC=0.5951 | AUCPR=0.1538
Released NIBS Cos-Max | seed=None | AUROC=0.7660 | AUCPR=0.2149
Released NIBS Cos-Mean | seed=None | AUROC=0.6580 | AUCPR=0.1905
Original formulas evaluated on ALL reviewed evaluation steps.


In [ ]:

# Released settings: failed training trajectories, Adam 1e-3,
# maximum 1000 epochs, training-loss monitor with patience 100.
R_MAX_EPOCHS = 1000
R_SEEDS = [0, 1, 2, 3, 4]

R_PAIR_CACHE = R_DIR / "mcs_edge_mappings"
R_PAIR_CACHE.mkdir(exist_ok=True)


def r_edge_mapping(query_id, reference_id):
    signature = {
        "commit": R_COMMIT,
        "data": DATA_HASH,
        "query": query_id,
        "reference": reference_id,
        "query_graph": r_data[query_id]["structure_str"],
        "reference_graph": r_data[reference_id]["structure_str"],
        "query_text": r_data[query_id]["reasoning_string"],
        "reference_text": r_data[reference_id]["reasoning_string"],
        "node_threshold": 0.7,
        "edge_threshold": 0.7,
    }
    key = hashlib.sha256(
        json.dumps(signature, sort_keys=True).encode()
    ).hexdigest()
    path = R_PAIR_CACHE / f"{key}.json"

    if path.exists():
        pairs = json.loads(path.read_text())
        return {tuple(a): tuple(b) for a, b in pairs}

    _, _, mapping = r_matcher.find_maximum_common_subgraph(
        r_graphs[query_id],
        r_graphs[reference_id],
        r_data[query_id]["reasoning_string"],
        r_data[reference_id]["reasoning_string"],
    )
    path.write_text(json.dumps([
        [list(a), list(b)] for a, b in mapping.items()
    ]))
    return mapping


# Precompute actual per-reference EDGE mappings—not node-frequency targets.
r_training_examples = []

for tid in tqdm(TRAIN_TASKS, desc="Original MCS edge supervision"):
    run_ids = [
        run["id"] for run in sims_by_task.get(tid, [])
        if run["id"] in r_data
    ]
    passing = [rid for rid in run_ids if r_success[rid]]
    failing = [rid for rid in run_ids if not r_success[rid]]

    for rid in failing:
        if passing:
            mappings = [
                r_edge_mapping(rid, ref) for ref in passing
            ]
            r_training_examples.append((rid, mappings))

assert r_training_examples, "No failed training trajectories with references"

print("Failed training trajectories:", len(r_training_examples))
print(
    "Trajectories with at least one nonempty MCS:",
    sum(any(mapping for mapping in maps)
        for _, maps in r_training_examples),
)

# Construct before seeding to avoid data preparation changing model seeds.
r_mcs_cache = r_gibs_module.MCSCache(
    str(R_DIR / "unused_feature_cache.json")
)

r_gibs_conf = {}

for seed in R_SEEDS:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = r_gibs_module.GIBErrorDetectionModel(
        embedding_dim=768,
        hidden_dim=128,
        mcs_finder=r_matcher,
        mcs_cache=r_mcs_cache,
    ).to(DEVICE)

    checkpoint = R_DIR / f"original_gibs_seed{seed}.pt"

    if checkpoint.exists():
        model.load_state_dict(torch.load(
            checkpoint, map_location=DEVICE, weights_only=True
        ))
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        monitor = r_gibs_module.ConvergenceMonitor(
            patience=100, min_delta=1e-4
        )

        best_loss = float("inf")
        best_state = None

        for epoch in range(R_MAX_EPOCHS):
            model.train()
            epoch_losses = []
            component_values = {}
            epoch_masks = []

            # Preserve task/run ordering, as in the released loop.
            for rid, mappings in r_training_examples:
                optimizer.zero_grad()
                outputs = model(r_data[rid], [])

                losses = model.compute_losses(
                    r_data[rid],
                    [],
                    outputs["edge_masks"],
                    None,
                    r_graphs[rid],
                    outputs["edge_ids"],
                    mappings,
                )
                loss = sum(losses.values())
                assert torch.isfinite(loss)

                if loss.item() > 0:
                    loss.backward()
                    optimizer.step()

                epoch_losses.append(float(loss.item()))

                for name, value in losses.items():
                    component_values.setdefault(name, []).append(
                        float(value.item())
                    )

                epoch_masks.extend(
                    outputs["edge_masks"].detach().cpu().numpy().ravel()
                )

            average_loss = float(np.mean(epoch_losses))
            component_means = {
                name: float(np.mean(values))
                for name, values in component_values.items()
            }

            monitor.update(
                epoch, average_loss, component_means, epoch_masks
            )

            if average_loss < best_loss:
                best_loss = average_loss
                best_state = copy.deepcopy(model.state_dict())

            if epoch % 10 == 0:
                print(
                    f"Released GIBS seed={seed}, epoch={epoch + 1}, "
                    f"loss={average_loss:.4f}",
                    flush=True,
                )

            if monitor.should_stop or monitor.is_converged():
                break

        assert best_state is not None
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), checkpoint)

    model.eval()
    confidence = {}

    with torch.no_grad():
        for rid in sorted({rid for rid, _ in paper_keys}):
            output = model(r_data[rid], [])
            graph = r_graphs[rid]

            for edge, score in zip(
                output["edge_ids"],
                output["edge_masks"].cpu().reshape(-1).tolist(),
            ):
                edge_name = graph.edges[edge]["label"]
                idx = r_edge_steps[rid][edge_name]
                confidence[(rid, idx)] = float(score)

    r_gibs_conf[seed] = confidence

    record_paper_result(
        "Released GIBS / original edge-MCS loss",
        confidence,
        seed,
    )
    del model

# Separate summary: do not mix reviewed labels with old-label results.
released_results = pd.DataFrame(PAPER_ROWS)
released_summary = released_results.groupby("method")[
    ["AUROC", "AUCPR", "first_error_AUROC"]
].agg(["mean", "std", "count"])

display(released_summary.round(4))
released_summary.to_csv(R_DIR / "reviewed_summary.csv")

# Additional failed-only evaluation: original NIBS skips successful runs.
failed_keys = [
    key for key in paper_keys if not r_success[key[0]]
]
failed_y = np.array([paper_labels[key] for key in failed_keys])

failed_rows = []
if len(np.unique(failed_y)) == 2:
    for method, confidence in r_nibs_conf.items():
        scores = np.array([1 - confidence[k] for k in failed_keys])
        failed_rows.append({
            "method": method,
            "seed": None,
            "n": len(failed_keys),
            "AUROC": roc_auc_score(failed_y, scores),
            "AUCPR": average_precision_score(failed_y, scores),
        })

    for seed, confidence in r_gibs_conf.items():
        scores = np.array([1 - confidence[k] for k in failed_keys])
        failed_rows.append({
            "method": "Released GIBS / original edge-MCS loss",
            "seed": seed,
            "n": len(failed_keys),
            "AUROC": roc_auc_score(failed_y, scores),
            "AUCPR": average_precision_score(failed_y, scores),
        })

    failed_results = pd.DataFrame(failed_rows)
    display(failed_results)
    failed_results.to_csv(R_DIR / "failed_only_results.csv", index=False)

print("Saved to:", R_DIR)


Original MCS edge supervision:   0%|          | 0/15 [00:00<?, ?it/s]